# Imports

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# %pip install scikit-learn

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy.sparse import coo_matrix
import random
import numpy as np
import pickle
import time
from scipy.special import expit
from pathlib import Path
import os

# Load Item Data

In [4]:
# import os
# os.listdir("/content/drive")

In [5]:
# os.getcwd()

# Load data

## User–Item interaction matrix

This matrix stores explicit ratings
Shape ∈R∣U∣×∣I∣

In [6]:
items_fashion_cleaned_df = pd.read_csv("Amazon_items_cleaned.csv")
ratings_cleaned_df = pd.read_csv( "Amazon_ratings_cleaned.csv")

In [7]:
# ratings_cleaned_df.shape

In [8]:
# ratings_cleaned_df.head(10)

In [9]:
user_ids = ratings_cleaned_df["user_id"].unique()
item_ids = ratings_cleaned_df["parent_asin"].unique()

user_to_idx = {u: i for i, u in enumerate(user_ids)}
item_to_idx = {i: j for j, i in enumerate(item_ids)}

idx_to_user = {i: u for u, i in user_to_idx.items()}
idx_to_item = {j: i for i, j in item_to_idx.items()}

## Load the coo_matrix

In [50]:
from scipy.sparse import load_npz
user_item_matrix = load_npz( "user_item_matrix.npz")

In [ ]:
# coo = user_item_matrix.tocoo()
# list(zip(coo.row[:10], coo.col[:10], coo.data[:10]))

In [12]:
# len(user_ids)

In [13]:
# user_item_matrix.shape

In [14]:
print("user item interation matrix shape:", user_item_matrix.shape)
print("user item interation matrix nnz (ratings):", user_item_matrix.nnz)

assert user_item_matrix.shape[0] == len(user_ids)
assert user_item_matrix.shape[1] == len(item_ids)
assert user_item_matrix.nnz == len(ratings_cleaned_df)

user item interation matrix shape: (1089923, 182207)
user item interation matrix nnz (ratings): 8528231


In [15]:
# items_fashion_cleaned_df.shape

## Item–Category information matrix


Items may belong to multiple categories → multiple 1's per row.

Shape∈{0,1}∣I∣×∣C∣

### Create category index mapping

In [16]:
# items_fashion_cleaned_df["categories_list_pruned"]

In [17]:
import ast

# All retained categories
# categories_list_pruned is stored as a string in CSV -> parse it back to a list
all_categories = sorted({
    cat
    for cats_raw in items_fashion_cleaned_df["categories_list_pruned"]
    for cat in (ast.literal_eval(cats_raw) if isinstance(cats_raw, str) else cats_raw)
})

category_to_idx = {c: i for i, c in enumerate(all_categories)}
idx_to_category = {i: c for c, i in category_to_idx.items()}

print(f"Number of categories: {len(all_categories)}")
print(f"Sample categories: {all_categories[:5]}")

In [18]:
# items_fashion_cleaned_df.head(5)

In [19]:
# print(len(all_categories))
# print(len(category_to_idx))
# type(idx_to_category)

### Build Item × Category matrix (multi-category safe)

In [20]:
import ast

cat_rows = []
cat_cols = []

for _, row in items_fashion_cleaned_df.iterrows():
    item_id = row["parent_asin"]

    # Skip items not in ratings matrix
    if item_id not in item_to_idx:
        continue

    item_idx = item_to_idx[item_id]

    # Parse categories — stored as string in CSV
    cats_raw = row["categories_list_pruned"]
    cats = ast.literal_eval(cats_raw) if isinstance(cats_raw, str) else cats_raw

    for cat in cats:
        cat_rows.append(item_idx)
        cat_cols.append(category_to_idx[cat])

item_category_matrix = coo_matrix(
    (np.ones(len(cat_rows)), (cat_rows, cat_cols)),
    shape=(len(item_to_idx), len(category_to_idx))
)

print(f"item_category_matrix shape: {item_category_matrix.shape}")
print(f"item_category_matrix nnz:   {item_category_matrix.nnz}")

### Sanity Checks

In [21]:
# print("item category information matrix shape:", item_category_matrix.shape)
# print("item category information matrix nnz (item–category links):", item_category_matrix.nnz)

# assert item_category_matrix.shape[0] == user_item_matrix.shape[1]     # same number of items
# assert item_category_matrix.nnz >= len(items_fashion_cleaned_df)

## Cross Matrix Alignment Checks

### Item alignment

In [22]:
# All items used in ratings
# items_in_user_item_matrix = set(item_ids)

# All items used in categories
# items_in_item_category_matrix = set(items_fashion_cleaned_df["parent_asin"])

# assert items_in_user_item_matrix.issubset(items_in_item_category_matrix), \
#     "Some items in user item matrix have no category metadata!"

### Index consistency

In [23]:
# random_item = random.choice(list(items_in_user_item_matrix))

# item_idx = item_to_idx[random_item]

# print("Item ASIN:", random_item)
# print("Categories:",
#       items_fashion_cleaned_df
#       .loc[items_fashion_cleaned_df["parent_asin"] == random_item,
#            "categories_list_pruned"]
#       .values[0])

# print("Nonzero category indices:",
#       item_category_matrix.getrow(item_idx).nonzero()[1])

# print("Category names:",
#       [idx_to_category[i] for i in item_category_matrix.getrow(item_idx).nonzero()[1]])

The printed categories match so the pipeline is correct.

### Check Density Sanity

In [24]:
# print("Avg ratings per user:", user_item_matrix.nnz / user_item_matrix.shape[0])
# print("Avg ratings per item:", user_item_matrix.nnz / user_item_matrix.shape[1])
# print("Avg categories per item:", item_category_matrix.nnz / item_category_matrix.shape[0])

In [ ]:
# from pathlib import Path


# np.save("user_to_idx.npy", user_to_idx)
# np.save("item_to_idx.npy", item_to_idx)

## To Load Later

In [26]:
# user_to_idx = np.load(BASE / "user_to_idx.npy", allow_pickle=True).item()
# item_to_idx = np.load(BASE / "item_to_idx.npy", allow_pickle=True).item()

## For now to delete from RAM

In [27]:
# del user_to_idx, item_to_idx
# import gc; gc.collect()

# Run MF

Create positive and negative indexes in the user-item interaction matrix for future use in the MF

In [28]:
# num_users_mf = user_item_matrix.shape[0]
# num_items_mf = user_item_matrix.shape[1]

# pos_ex = {}
# neg_ex = {}
# pos_ex_num = {}

# for user_idx in range(num_users_mf):
#     # Get the sparse row for the current user
#     user_row = user_item_matrix.getrow(user_idx)

#     # Get column indices of positive interactions (non-zero values)
#     positive_item_indices = user_row.nonzero()[1].tolist()
#     pos_ex[user_idx] = positive_item_indices
#     pos_ex_num[user_idx] = len(positive_item_indices)

#     # Get all possible item indhgbices
#     all_item_indices = set(range(num_items_mf))
#     # Get item indices of negative interactions
#     negative_item_indices = list(all_item_indices - set(positive_item_indices))
#     neg_ex[user_idx] = negative_item_indices

In [29]:
# user_item_matrix.shape

# Trial (new code for MF)
Preparing data smaples

Positive sampling

In [51]:
import numpy as np
from scipy.special import expit

# CSR matrix
R = user_item_matrix

users_pos, items_pos = R.nonzero()
num_users, num_items = R.shape

Negative sampling

In [ ]:
def sample_negatives(R, num_neg=1):
    """
    # For each positive interaction, sample num_neg negative items.
    """
    user_neg = []
    item_neg = []

    num_items = R.shape[1]

    users_pos = np.unique(R.nonzero()[0])

    for u in users_pos:
        pos_items = set(R[u].indices)

        cnt = 0
        target = num_neg * len(pos_items)

        while cnt < target:
            j = np.random.randint(num_items)
            if j not in pos_items:
                user_neg.append(u)
                item_neg.append(j)
                cnt += 1

    return np.array(user_neg), np.array(item_neg)

# MF class

In [ ]:
import numpy as np
from scipy.special import expit

class SparseMF:
    def __init__(self, num_users, num_items, K=20, lr=0.01, reg=0.01):
        self.K = K
        self.lr = lr
        self.reg = reg

        self.P = np.random.normal(0, 0.1, (num_users, K))
        self.Q = np.random.normal(0, 0.1, (num_items, K))
        self.b_u = np.zeros(num_users)
        self.b_i = np.zeros(num_items)

    def train(self, users, items, labels, epochs=10, batch_size=1024):
        n = len(users)
    
        for epoch in range(epochs):
            perm = np.random.permutation(n)
            users_shuf = users[perm]
            items_shuf = items[perm]
            labels_shuf = labels[perm]
    
            for start in range(0, n, batch_size):
                end = min(start + batch_size, n)
                u = users_shuf[start:end]
                i = items_shuf[start:end]
                y = labels_shuf[start:end]
    
                Pu = self.P[u].copy()
                Qi = self.Q[i].copy()
    
                score = self.b_u[u] + self.b_i[i] + np.sum(Pu * Qi, axis=1)
                pred = expit(score)
    
                err = y - pred
    
                grad_bu = err - self.reg * self.b_u[u]
                grad_bi = err - self.reg * self.b_i[i]
                grad_P  = err[:, None] * Qi - self.reg * Pu
                grad_Q  = err[:, None] * Pu - self.reg * Qi
    
                np.add.at(self.b_u, u, self.lr * grad_bu)
                np.add.at(self.b_i, i, self.lr * grad_bi)
                np.add.at(self.P,   u, self.lr * grad_P)
                np.add.at(self.Q,   i, self.lr * grad_Q)

    
            print(f"Epoch {epoch+1} done")


    # 🔹 RAW score (for AUC & ranking)
    def predict(self, users, items):
        return (
            self.b_u[users] +
            self.b_i[items] +
            np.sum(self.P[users] * self.Q[items], axis=1)
        )

    # 🔹 Full-item prediction for ONE user (for top-K)
    def predict_user(self, user_id):
        return (
            self.b_u[user_id] +
            self.b_i +
            self.P[user_id] @ self.Q.T
        )

# Training Data

In [33]:
# Positive samples
# users_p, items_p = R.nonzero()
# labels_p = np.ones(len(users_p))

# Negative samples
# users_n, items_n = sample_negatives(R, num_neg=1)
# labels_n = np.zeros(len(users_n))

# Combine
# users = np.concatenate([users_p, users_n])
# items = np.concatenate([items_p, items_n])
# labels = np.concatenate([labels_p, labels_n])

In [52]:
# =========================================
# CLEAN DATA PIPELINE FOR IMPLICIT MF
# =========================================

import numpy as np
from sklearn.model_selection import train_test_split
from scipy.sparse import csr_matrix
from collections import defaultdict

# -----------------------------------------
# Step 1: Get POSITIVES ONLY from R
# -----------------------------------------
users_pos, items_pos = R.nonzero()

# -----------------------------------------
# Step 2: Select 400000 users (from positives)
# -----------------------------------------
np.random.seed(42)

all_users = np.unique(users_pos)
selected_users = np.random.choice(all_users, size=min(400000, len(all_users)), replace=False)

mask = np.isin(users_pos, selected_users)

users_pos_sub = users_pos[mask]
items_pos_sub = items_pos[mask]

print("Using positives:", len(users_pos_sub))
print("Using users:", len(np.unique(users_pos_sub)))

# -----------------------------------------
# Step 3: Remap users to 0..N-1 (keep item ids)
# -----------------------------------------
sub_users_unique = np.unique(users_pos_sub)
user2id = {u: i for i, u in enumerate(sub_users_unique)}

users_pos_sub_ids = np.array([user2id[u] for u in users_pos_sub])
items_pos_sub_ids = items_pos_sub

num_users = len(sub_users_unique)
num_items = R.shape[1]

print("num_users =", num_users)
print("num_items =", num_items)

# -----------------------------------------
# Step 4: Split POSITIVES ONLY into train / val
# -----------------------------------------
users_tr_pos, users_val_pos, items_tr_pos, items_val_pos = train_test_split(
    users_pos_sub_ids,
    items_pos_sub_ids,
    test_size=0.2,
    random_state=42
)

print("Train positives:", len(users_tr_pos))
print("Val positives:", len(users_val_pos))

# -----------------------------------------
# Step 5: Negative sampling function
# -----------------------------------------
def sample_negatives_for_pairs(users_pos, items_pos, R, num_neg=1):
    num_items = R.shape[1]

    user_neg = []
    item_neg = []

    pos_by_user = defaultdict(set)
    for u, i in zip(users_pos, items_pos):
        pos_by_user[u].add(i)

    for u, pos_items in pos_by_user.items():
        target = num_neg * len(pos_items)
        cnt = 0
        while cnt < target:
            j = np.random.randint(num_items)
            if j not in pos_items:
                user_neg.append(u)
                item_neg.append(j)
                cnt += 1

    return np.array(user_neg), np.array(item_neg)

# -----------------------------------------
# Step 6: Sample negatives SEPARATELY
# -----------------------------------------
users_tr_neg, items_tr_neg = sample_negatives_for_pairs(
    users_tr_pos, items_tr_pos, R, num_neg=1
)

users_val_neg, items_val_neg = sample_negatives_for_pairs(
    users_val_pos, items_val_pos, R, num_neg=1
)

print("Train negatives:", len(users_tr_neg))
print("Val negatives:", len(users_val_neg))

# -----------------------------------------
# Step 7: Build final train / val sets
# -----------------------------------------
users_tr = np.concatenate([users_tr_pos, users_tr_neg])
items_tr = np.concatenate([items_tr_pos, items_tr_neg])
y_tr = np.concatenate([np.ones(len(users_tr_pos)), np.zeros(len(users_tr_neg))])

users_val = np.concatenate([users_val_pos, users_val_neg])
items_val = np.concatenate([items_val_pos, items_val_neg])
y_val = np.concatenate([np.ones(len(users_val_pos)), np.zeros(len(users_val_neg))])

# Shuffle
perm = np.random.permutation(len(users_tr))
users_tr, items_tr, y_tr = users_tr[perm], items_tr[perm], y_tr[perm]

perm = np.random.permutation(len(users_val))
users_val, items_val, y_val = users_val[perm], items_val[perm], y_val[perm]

print("Final train size:", len(users_tr))
print("Final val size:", len(users_val))

print("y_tr mean:", float(y_tr.mean()))
print("y_val mean:", float(y_val.mean()))

# -----------------------------------------
# Step 8: Build R_train and R_val (for Top-K eval)
# -----------------------------------------
R_train = csr_matrix(
    (np.ones(len(users_tr_pos)), (users_tr_pos, items_tr_pos)),
    shape=(num_users, num_items)
)

R_val = csr_matrix(
    (np.ones(len(users_val_pos)), (users_val_pos, items_val_pos)),
    shape=(num_users, num_items)
)

print("R_train shape:", R_train.shape)
print("R_val shape:", R_val.shape)

Using positives: 3133692
Using users: 400000
num_users = 400000
num_items = 182207
Train positives: 2506953
Val positives: 626739
Train negatives: 2506953
Val negatives: 626739
Final train size: 5013906
Final val size: 1253478
y_tr mean: 0.5
y_val mean: 0.5
R_train shape: (400000, 182207)
R_val shape: (400000, 182207)


In [ ]:
# print("Pos samples:", len(users_p))
# print("Neg samples:", len(users_n))

In [ ]:
# print("y_tr mean:", float(y_tr.mean()))
# print("y_val mean:", float(y_val.mean()))

In [ ]:
# from sklearn.metrics import roc_auc_score

# mf = SparseMF(num_users, num_items, K=20, lr=0.005, reg=0.0001)

# scores0 = mf.predict(users_val, items_val)
# print("AUC before:", roc_auc_score(y_val, scores0))

# mf.train(users_tr, items_tr, y_tr, epochs=20)

# scores1 = mf.predict(users_val, items_val)
# print("AUC after:", roc_auc_score(y_val, scores1))

In [38]:
# print("OK")

# Training!

In [39]:
# print("=== TRAINING START CHECK ===")
# print(f"Users range: [{users_tr.min()}, {users_tr.max()}]")
# print(f"Items range: [{items_tr.min()}, {items_tr.max()}]")
# print(f"Labels mean (pos ratio): {labels.mean():.4f}")
# print(f"Num samples: {len(labels)}")
# print(f"Embedding users: {num_users}, items: {num_items}")

In [40]:
num_users_tr = num_users          # from the split cell (should be 4000)
num_items_tr = num_items 

In [41]:
mf = SparseMF(num_users_tr, num_items_tr, K=20, lr=0.005, reg=0.01)
mf.train(users_tr, items_tr, y_tr, epochs=10)

Epoch 1 done
Epoch 2 done
Epoch 3 done
Epoch 4 done
Epoch 5 done
Epoch 6 done
Epoch 7 done


KeyboardInterrupt: 

In [ ]:
import pickle

with open("mf_model.pkl", "wb") as f:
    pickle.dump(mf, f)
print("Saved: mf_model.pkl")

In [ ]:
# def l2_norm(x):
#     return float(np.sqrt((x * x).sum()))

# P0 = mf.P.copy()
# Q0 = mf.Q.copy()
# bu0 = mf.b_u.copy()
# bi0 = mf.b_i.copy()

# mf.train(users_tr, items_tr, y_tr, epochs=1)

# print("dP:", l2_norm(mf.P - P0))
# print("dQ:", l2_norm(mf.Q - Q0))
# print("dbu:", l2_norm(mf.b_u - bu0))
# print("dbi:", l2_norm(mf.b_i - bi0))

# Recommendations

In [ ]:
# print("Model num_items:", num_items_tr)
# print("R_val num_items:", R_val.shape[1])

In [ ]:
def recommend(mf, user_idx, top_k=10, seen=None):
    scores = mf.b_u[user_idx] + mf.b_i + mf.P[user_idx] @ mf.Q.T
    if seen is not None:
        scores[list(seen)] = -np.inf
    return np.argsort(scores)[-top_k:][::-1]

In [ ]:
# u = 0
# seen_items = set(R_train[u].indices)   # ONLY training items
# top_items = recommend(mf, u, top_k=10, seen=seen_items)

In [ ]:
# top_items

# Evaluation

In [ ]:
# from sklearn.metrics import roc_auc_score
# from sklearn.metrics import log_loss


# scores = mf.predict(users_val, items_val)
# auc = roc_auc_score(y_val, scores)
# print(f"AUC: {auc:.4f}")



# probs = 1 / (1 + np.exp(-scores))  # sigmoid if needed
# ll = log_loss(y_val, probs)
# print(f"Log loss: {ll:.4f}")


# pred = (scores > 0).astype(int)
# acc = (pred == y_val).mean()
# print(f"Accuracy: {acc:.4f}")

Ranking evaluation

In [ ]:
## Create R_val

# from scipy.sparse import coo_matrix

# Keep only POSITIVE interactions
# mask = y_val == 1

# R_val = coo_matrix(
#     (np.ones(mask.sum()), (users_val[mask], items_val[mask])),
#     shape=(num_users, num_items)
# ).tocsr()

In [ ]:
# def recommend_top_k(mf, R, user_id, K=10):
#     seen_items = set(R[user_id].indices)
#     scores = mf.predict_user(user_id)

#     scores[list(seen_items)] = -np.inf
#     return np.argsort(scores)[-K:][::-1]

In [ ]:
# import os
# import numpy as np
# import pickle

# def precision_recall_at_k(mf, R_val, users, K=10, save_every=10_000, out_dir="pr_checkpoints"):
#     os.makedirs(out_dir, exist_ok=True)

#     precisions, recalls = [], []

#     job_id = os.environ.get("SLURM_JOB_ID", "no_job_id")
#     log_file = f"logging_{job_id}.txt"

#     with open(log_file, "a") as f:
#         f.write(f"JOB_ID={job_id}\n")

#     total_users = len(users)

#     for idx, u in enumerate(users, start=1):
#         true_items = set(R_val[u].indices)
#         if idx == 1 or idx % save_every == 0:
#             with open(log_file, "a") as f:
#                 f.write(f"logged user #{idx}/ {total_users}\n")

#         if len(true_items) == 0:
#             continue

#         recs = set(recommend_top_k(mf, R_train, u, K))
#         hits = len(recs & true_items)

        # print("val items:", R_val[u].indices[:10])
        # print("recs:", list(recs))
        # print("hits:", set(recs) & set(R_val[u].indices))



#         precisions.append(hits / K)
#         recalls.append(hits / len(true_items))

        # Save snapshot every save_every users
#         if idx % save_every == 0:
#             p_now = float(np.mean(precisions))
#             r_now = float(np.mean(recalls))

#             ckpt_file = os.path.join(out_dir, f"pr_at_{idx}.pkl")
#             with open(ckpt_file, "wb") as f:
#                 pickle.dump({
#                     "num_users": idx,
#                     "precision": p_now,
#                     "recall": r_now
#                 }, f)

#             print(f"Saved snapshot at {idx}: P={p_now:.4f}, R={r_now:.4f}")

    # Final snapshot
#     p_final = float(np.mean(precisions))
#     r_final = float(np.mean(recalls))

#     ckpt_file = os.path.join(out_dir, f"pr_at_{total_users}.pkl")
#     with open(ckpt_file, "wb") as f:
#         pickle.dump({
#             "num_users": total_users,
#             "precision": p_final,
#             "recall": r_final
#         }, f)

#     return p_final, r_final


In [ ]:
# users_eval = np.unique(users_val)
# p, r = precision_recall_at_k(mf, R_val, users_eval, K=10)
# print(f"Precision@10: {p:.4f}")
# print(f"Recall@10: {r:.4f}")

NDCG@K

In [ ]:
# import numpy as np

# def ndcg_at_k(mf, R_val, users, K=10):
#     ndcgs = []

#     for u in users:
#         true_items = set(R_val[u].indices)
#         if not true_items:
#             continue

#         recs = recommend_top_k(mf, R_val, u, K)

#         dcg = 0.0
#         for i, item in enumerate(recs):
#             if item in true_items:
#                 dcg += 1 / np.log2(i + 2)

#         idcg = sum(1 / np.log2(i + 2) for i in range(min(len(true_items), K)))
#         ndcgs.append(dcg / idcg if idcg > 0 else 0)

#     return np.mean(ndcgs)

In [ ]:
# ndcg = ndcg_at_k(mf, R_val, users_eval, K=10)
# print(f"NDCG@10: {ndcg:.4f}")

In [ ]:
print("OK")

## Matryoshka Sparse Autoencoder (SAE) - New Cells

In [ ]:
# Sampling: 100-row mini version of the data for quick sanity checks
# import numpy as np
# import torch
# from torch.utils.data import DataLoader

# Sample 100 rows from the original ratings dataframe
# df_sample = ratings_cleaned_df.sample(n=100, random_state=42).reset_index(drop=True)

# Build local int-index mappings for the sample
# sample_user_list = list(df_sample["user_id"].unique())
# sample_item_list = list(df_sample["parent_asin"].unique())
# su2i = {u: i for i, u in enumerate(sample_user_list)}
# si2i = {it: j for j, it in enumerate(sample_item_list)}

# n_users_s = len(sample_user_list)
# n_items_s = len(sample_item_list)

# Train a tiny MF on the sample (use the existing SparseMF class)
# users_s = np.array([su2i[u] for u in df_sample["user_id"]])
# items_s = np.array([si2i[i] for i in df_sample["parent_asin"]])
# labels_s = np.ones(len(users_s))
# mf_sample = SparseMF(n_users_s, n_items_s, K=20, lr=0.01, reg=0.01)
# mf_sample.train(users_s, items_s, labels_s, epochs=3)

# Build the mf_recommender object the SAE cells expect
# mf_recommender = mf_sample

# Build tensors + DataLoaders
# user_tensor = torch.tensor(mf_recommender.P, dtype=torch.float32)
# item_tensor = torch.tensor(mf_recommender.Q, dtype=torch.float32)
# dataset_users = user_tensor
# dataset_items = item_tensor
# user_loader = DataLoader(dataset_users, batch_size=32, shuffle=True)
# item_loader = DataLoader(dataset_items, batch_size=32, shuffle=True)

# print(f"Sample df rows     : {len(df_sample)}")
# print(f"Unique sample users: {n_users_s}")
# print(f"Unique sample items: {n_items_s}")
# print(f"user_tensor shape  : {user_tensor.shape}")
# print(f"item_tensor shape  : {item_tensor.shape}")

In [ ]:
# print('K' in dir(), 'num_users' in dir(), 'num_items' in dir())

In [ ]:
# print(num_users, num_items)

In [ ]:
# print([k for k in dir() if 'auto' in k.lower() or 'sae' in k.lower() or 'encoder' in k.lower()])

In [ ]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# from torch.utils.data import DataLoader

# # Import your existing components from the notebook
# from your_notebook import Autoencoder           # base autoencoder class
# from your_notebook import beta_schedule         # dynamic beta schedule function
# from your_notebook import kl_divergence_loss    # KL divergence sparsity loss

# -----------------------------------
# 1) Define Matryoshka Sparse Autoencoder with Batch‐TopK sparsity
# -----------------------------------
# class MatryoshkaAutoencoder(Autoencoder):
#     def __init__(
#         self,
#         latent_dim: int,
#         input_dim: int,
#         group_sizes: list[int],
#         k: int,
#         threshold_beta: float = 0.999,
#         activation: nn.Module = nn.ReLU(),
#         tied: bool = True,
#         normalize: bool = True
#     ):
#         """
#         Matryoshka SAE with:
#           - nested prefixes (group_sizes)
#           - a learnable threshold for sparsity
#           - global batch‐topK masking (k nonzeros per sample)
#         """
#         super().__init__(latent_dim, input_dim,
#                          activation=activation,
#                          tied=tied, normalize=normalize)
#         assert group_sizes[-1] == latent_dim, "last prefix must equal latent_dim"
#         self.group_sizes = group_sizes
#         self.k = k
#         self.threshold_beta = threshold_beta
        # threshold < 0 indicates "not yet initialized"
#         self.register_buffer('threshold', torch.tensor(-1.0))

#     def encode(self, x: torch.Tensor):
        # preprocess + encode pre‐activation
#         x_norm, info = self.preprocess(x)
#         z_pre = self.encode_pre_act(x_norm)
#         z = self.activation(z_pre)  # [B, D]

        # 1) update dynamic threshold based on current activations
#         act = z[z > 0]
#         if act.numel() > 0:
#             min_act = act.min().detach()
#             if self.threshold < 0:
#                 self.threshold = min_act
#             else:
#                 self.threshold = (self.threshold_beta * self.threshold
#                                   + (1 - self.threshold_beta) * min_act)

        # 2) apply threshold mask
#         z_thresh = z * (z > self.threshold)

        # 3) flatten and take top-(k * B) activations across the batch
#         B, D = z_thresh.shape
#         flat = z_thresh.view(-1)
#         topk = flat.topk(self.k * B, sorted=False)
#         mask = torch.zeros_like(flat)
#         mask[topk.indices] = topk.values
#         f = mask.view(B, D)

#         return f, info

#     def forward(self, x: torch.Tensor):
#         """
#         Returns:
#           f      = sparse code [B, D]
#           recons = list of reconstructions [B, input_dim] for each prefix
#         """
#         f, info = self.encode(x)
#         recons = []
#         for m in self.group_sizes:
#             f_m = f.clone()
#             f_m[:, m:] = 0          # zero out units beyond prefix m
#             x_hat = self.decode(f_m, info)
#             recons.append(x_hat)
#         return f, recons

# -----------------------------------
# 2) Prepare DataLoaders for MF outputs
# -----------------------------------
# Assume you have two numpy arrays or tensors:
#   user_embeddings: shape [N_users, K]
#   item_embeddings: shape [N_items, K]
# user_tensor = torch.tensor(mf_recommender.P, dtype=torch.float32)
# item_tensor = torch.tensor(mf_recommender.Q, dtype=torch.float32)

# user_loader = DataLoader(user_tensor, batch_size=256, shuffle=True)
# item_loader = DataLoader(item_tensor, batch_size=256, shuffle=True)

# -----------------------------------
# 3) Train function for Matryoshka SAE
# -----------------------------------
# def train_matryoshka(
#     sae: MatryoshkaAutoencoder,
#     user_loader: DataLoader,
#     item_loader: DataLoader,
#     num_epochs: int = 30,
#     lr: float = 1e-3,
#     mse_weight: float = 8.0,
#     inner_start: float = 0.0,
#     inner_end: float = 20.0,
#     warmup: int = 5,
#     device: str = 'cuda'
# ):
#     """
#     Trains Matryoshka SAE with:
#       1) Nested reconstruction losses per prefix
#       2) Batch‐TopK sparsity (no L1/KL penalties needed)
#       3) Dynamic inner‐product loss on final reconstruction
#     """
    # sae.to(device)
#     optimizer = torch.optim.Adam(sae.parameters(), lr=lr)

#     for epoch in range(1, num_epochs + 1):
#         epoch_loss = 0.0

#         for u, i in zip(user_loader, item_loader):
            # u = u.to(device)
            # i = i.to(device)

            # encode + reconstruct
#             f_u, recons_u = sae(u)
#             f_i, recons_i = sae(i)

            # 1) nested reconstruction loss (sum over all prefix levels)
#             recon_u = sum(F.mse_loss(r, u) for r in recons_u)
#             recon_i = sum(F.mse_loss(r, i) for r in recons_i)
#             recon_term = mse_weight * (recon_u + recon_i)

            # 2) dynamic inner‐product loss on highest‐level reconstruction
#             u_hat = recons_u[-1]
#             i_hat = recons_i[-1]
#             inner_orig = u @ i.T
#             inner_recon = u_hat @ i_hat.T
#             beta = beta_schedule(
#                 epoch - 1, num_epochs,
#                 beta_start=inner_start,
#                 beta_end=inner_end,
#                 warmup=warmup
#             )
#             inner_term = beta * F.mse_loss(inner_recon, inner_orig)

            # total loss
#             loss = recon_term + inner_term

#             optimizer.zero_grad()
#             loss.backward()
#             optimizer.step()

#             epoch_loss += loss.item() * u.size(0)

#         avg_loss = epoch_loss / len(user_loader.dataset)
#         print(f"Epoch {epoch}/{num_epochs} — avg_loss: {avg_loss:.4f}")

In [6]:
# ============================================================
# Base classes and helper functions required by MatryoshkaAutoencoder.
# Source: ML1M_SAE_MF_1902.ipynb (cells 31, 34, 35) + beta_schedule
# from Copy_of_lastFM_SAE_NCF_111125.ipynb.
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import Any, Callable


# ------------------------------------------------------------------
# Layer Norm helper (exact from ML1M cell 34)
# ------------------------------------------------------------------
def LN(x: torch.Tensor, eps: float = 1e-5) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    if type(x) == np.ndarray:
        x = torch.from_numpy(x)
    mu = x.mean(dim=-1, keepdim=True)
    x = x - mu
    std = x.std(dim=-1, keepdim=True)
    x = x / (std + eps)
    return x, mu, std


# ------------------------------------------------------------------
# Tensor utilities (exact from ML1M cell 31)
# ------------------------------------------------------------------
def normalize_matrix(matrix):
    """Min-max normalize per column; returns numpy array (NaN -> 0)."""
    matrix = matrix.float()
    min_val, _ = matrix.min(axis=0)
    max_val, _ = matrix.max(axis=0)
    normalized_matrix = (matrix - min_val) / (max_val - min_val)
    return np.nan_to_num(normalized_matrix)


def convert_to_dense_tensor(sparse_matrix):
    """Convert a sparse matrix to a dense float32 torch Tensor."""
    dense_matrix = np.array(sparse_matrix)
    dense_tensor = torch.tensor(dense_matrix, dtype=torch.float32)
    return dense_tensor


def pad_or_truncate_tensor(tensor1, target_dim):
    """
    Pad (right-side zeros) or truncate the FEATURE dimension of a 2-D batch
    tensor so that tensor1.shape[1] == target_dim.
    """
    if not isinstance(tensor1, torch.Tensor):
        tensor1 = torch.tensor(tensor1, dtype=torch.float32)
    flattened_tensor = tensor1.reshape(tensor1.shape[0], -1)
    if flattened_tensor.shape[1] < target_dim:
        pad_size = target_dim - flattened_tensor.shape[1]
        padded_tensor = torch.nn.functional.pad(flattened_tensor, (0, pad_size))
    else:
        padded_tensor = flattened_tensor[:, :target_dim]
    return padded_tensor


def pad_or_truncate_tensor_0(tensor1, target_dim):
    """Older pad/truncate variant kept for backward-compatibility."""
    flattened_tensor = tensor1.reshape(tensor1.shape[0], -1)
    if flattened_tensor.shape[0] < target_dim:
        padded_tensor = torch.nn.functional.pad(flattened_tensor, (0, target_dim - flattened_tensor.size(1)))
    else:
        padded_tensor = flattened_tensor[:, :target_dim]
    return padded_tensor


def _pad_or_truncate_last_dim(x: torch.Tensor, target_size: int) -> torch.Tensor:
    """
    Internal helper used by Autoencoder.encode_pre_act: pads/truncates the
    LAST dimension of an arbitrary-rank tensor.
    """
    current_size = x.shape[-1]
    if current_size == target_size:
        return x
    elif current_size < target_size:
        pad = torch.zeros(*x.shape[:-1], target_size - current_size,
                          device=x.device, dtype=x.dtype)
        return torch.cat([x, pad], dim=-1)
    else:
        return x[..., :target_size]


def init_weights(m):
    """Xavier-uniform init for Linear layers."""
    if isinstance(m, nn.Linear):
        torch.nn.init.xavier_uniform_(m.weight)


# ------------------------------------------------------------------
# Tied decoder (exact from ML1M cell 34)
# ------------------------------------------------------------------
class TiedTranspose(nn.Module):
    def __init__(self, linear: nn.Linear):
        super().__init__()
        self.linear = linear

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        assert self.linear.bias is None
        return F.linear(x, self.linear.weight.t(), None)

    @property
    def weight(self) -> torch.Tensor:
        return self.linear.weight.t()

    @property
    def bias(self) -> torch.Tensor:
        return self.linear.bias


# ------------------------------------------------------------------
# TopK activation (exact from ML1M cell 34)
# ------------------------------------------------------------------
class TopK(nn.Module):
    def __init__(self, k: int, postact_fn: Callable = nn.ReLU()) -> None:
        super().__init__()
        self.k = k
        self.postact_fn = postact_fn

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        topk = torch.topk(x, k=self.k, dim=-1)
        values = self.postact_fn(topk.values)
        result = torch.zeros_like(x)
        result.scatter_(-1, topk.indices, values)
        return result

    def state_dict(self, destination=None, prefix="", keep_vars=False):
        state_dict = super().state_dict(destination, prefix, keep_vars)
        state_dict.update({prefix + "k": self.k,
                           prefix + "postact_fn": self.postact_fn.__class__.__name__})
        return state_dict

    @classmethod
    def from_state_dict(cls, state_dict: dict[str, torch.Tensor], strict: bool = True) -> "TopK":
        k = state_dict["k"]
        postact_fn = ACTIVATIONS_CLASSES[state_dict["postact_fn"]]()
        return cls(k=k, postact_fn=postact_fn)


ACTIVATIONS_CLASSES = {
    "ReLU": nn.ReLU,
    "Identity": nn.Identity,
    "TopK": TopK,
}


# ------------------------------------------------------------------
# Test-split placeholders
# ------------------------------------------------------------------
test_subset_users = []
test_subset_items = []
test_flag = 0


# ------------------------------------------------------------------
# Autoencoder base class (exact from ML1M cell 34)
# ------------------------------------------------------------------
class Autoencoder(nn.Module):
    def __init__(
        self, n_latents: int, n_inputs: int, activation: Callable = nn.ReLU(),
        tied: bool = False, normalize: bool = False
    ) -> None:
        super().__init__()
        self.pre_bias = nn.Parameter(torch.zeros(n_inputs))
        self.encoder: nn.Module = nn.Linear(n_inputs, n_latents, bias=False)
        self.latent_bias = nn.Parameter(torch.zeros(n_latents))
        self.activation = activation
        if tied:
            self.decoder: nn.Linear | TiedTranspose = TiedTranspose(self.encoder)
        else:
            self.decoder = nn.Linear(n_latents, n_inputs, bias=False)
        self.normalize = normalize
        self.loss = []
        self.test_subset_users_ind = test_subset_users
        self.test_subset_items_ind = test_subset_items
        self.weights_loss = [1, 0.0001, 0, 0]
        self.test = test_flag

    def encode_pre_act(self, x: torch.Tensor, latent_slice: slice = slice(None)) -> torch.Tensor:
        if type(x) == np.ndarray:
            x = torch.from_numpy(x)
        x = _pad_or_truncate_last_dim(x, self.pre_bias.shape[0])
        x = x - self.pre_bias
        latents_pre_act = F.linear(
            x, self.encoder.weight[latent_slice], self.latent_bias[latent_slice]
        )
        return latents_pre_act

    def preprocess(self, x: torch.Tensor) -> tuple[torch.Tensor, dict[str, Any]]:
        if not self.normalize:
            return x, dict()
        x, mu, std = LN(x)
        return x, dict(mu=mu, std=std)

    def encode(self, x: torch.Tensor) -> tuple[torch.Tensor, dict[str, Any]]:
        x, info = self.preprocess(x)
        return self.activation(self.encode_pre_act(x)), info

    def decode(self, latents: torch.Tensor, info: dict[str, Any] | None = None) -> torch.Tensor:
        ret = self.decoder(latents) + self.pre_bias
        if self.normalize:
            assert info is not None
            ret = ret * info["std"] + info["mu"]
        return ret

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        x, info = self.preprocess(x)
        latents_pre_act = self.encode_pre_act(x)
        latents = self.activation(latents_pre_act)
        recons = self.decode(latents, info)
        return latents_pre_act, latents, recons

    @classmethod
    def from_state_dict(
        cls, state_dict: dict[str, torch.Tensor], strict: bool = True
    ) -> "Autoencoder":
        n_latents, d_model = state_dict["encoder.weight"].shape
        activation_class_name = state_dict.pop("activation", "ReLU")
        activation_class = ACTIVATIONS_CLASSES.get(activation_class_name, nn.ReLU)
        normalize = activation_class_name == "TopK"
        activation_state_dict = state_dict.pop("activation_state_dict", {})
        if hasattr(activation_class, "from_state_dict"):
            activation = activation_class.from_state_dict(activation_state_dict, strict=strict)
        else:
            activation = activation_class()
            if hasattr(activation, "load_state_dict"):
                activation.load_state_dict(activation_state_dict, strict=strict)
        autoencoder = cls(n_latents, d_model, activation=activation, normalize=normalize)
        autoencoder.load_state_dict(state_dict, strict=strict)
        return autoencoder

    def state_dict(self, destination=None, prefix="", keep_vars=False):
        sd = super().state_dict(destination, prefix, keep_vars)
        sd[prefix + "activation"] = self.activation.__class__.__name__
        if hasattr(self.activation, "state_dict"):
            sd[prefix + "activation_state_dict"] = self.activation.state_dict()
        return sd


# ------------------------------------------------------------------
# KL-divergence sparsity loss (exact from ML1M cell 35)
# ------------------------------------------------------------------
def kl_divergence_loss(latent_activations, sparsity_target=0.05, eps=1e-6):
    """KL divergence between mean batch activation and target sparsity."""
    rho_hat = torch.clamp(torch.mean(latent_activations, dim=0), eps, 1 - eps)

    if (sum(rho_hat > 1) > 0) or (sum(rho_hat < 0) > 0):
        print(rho_hat)

    rho = torch.tensor(sparsity_target, dtype=torch.float32, device=latent_activations.device)
    kl_div = (rho * torch.log((rho + eps) / (rho_hat + eps))
              + (1 - rho) * torch.log(((1 - rho) + eps) / ((1 - rho_hat) + eps)))
    return torch.sum(kl_div)


# ------------------------------------------------------------------
# Beta schedule — used by train_matryoshka to ramp inner-product loss weight
# ------------------------------------------------------------------
def beta_schedule(epoch: int, max_epochs: int,
                  beta_start: float = 0.0, beta_end: float = 10.0,
                  warmup: int = 13) -> float:
    """Linear beta ramp: hold at beta_start for <warmup> epochs, then ramp to beta_end."""
    if epoch < warmup:
        return beta_start
    progress = (epoch - warmup) / max(1, max_epochs - warmup)
    return beta_start + progress * (beta_end - beta_start)


print("Defined: LN, normalize_matrix, convert_to_dense_tensor, "
      "pad_or_truncate_tensor, pad_or_truncate_tensor_0, _pad_or_truncate_last_dim, "
      "init_weights, TiedTranspose, TopK, ACTIVATIONS_CLASSES, "
      "Autoencoder, kl_divergence_loss, beta_schedule.")

Defined: LN, normalize_matrix, convert_to_dense_tensor, pad_or_truncate_tensor, pad_or_truncate_tensor_0, _pad_or_truncate_last_dim, init_weights, TiedTranspose, TopK, ACTIVATIONS_CLASSES, Autoencoder, kl_divergence_loss, beta_schedule.


In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader


# -----------------------------------
# 1) Define Matryoshka Sparse Autoencoder
# -----------------------------------
class MatryoshkaAutoencoder(Autoencoder):
    def __init__(self,
                 latent_dim: int,
                 input_dim: int,
                 group_sizes: list[int],
                 activation: nn.Module = nn.ReLU(),
                 tied: bool = True,
                 normalize: bool = True):
        super().__init__(latent_dim, input_dim,
                         activation=activation,
                         tied=tied, normalize=normalize)
        assert group_sizes[-1] == latent_dim, "the last prefix must equal latent_dim"
        self.group_sizes = group_sizes

    def forward(self, x: torch.Tensor):
        x_norm, info = self.preprocess(x)
        z_pre = self.encode_pre_act(x_norm)
        z = self.activation(z_pre)

        recons = []
        for m in self.group_sizes:
            z_masked = z.clone()
            z_masked[:, m:] = 0
            x_hat = self.decode(z_masked, info)
            recons.append(x_hat)

        return z_pre, z, recons


# -----------------------------------
# 2) Train function for Matryoshka SAE
# -----------------------------------
def train_matryoshka(
    sae: MatryoshkaAutoencoder,
    user_loader: DataLoader,
    item_loader: DataLoader,
    num_epochs: int = 30,
    lr: float = 1e-3,
    mse_weight: float = 8.0,
    l1_weight: float = 0.3,
    kl_weight: float = 0.003,
    inner_start: float = 0.0,
    inner_end: float = 20.0,
    warmup: int = 5,
    sparsity_target: float = 0.1,
    device: str = 'cuda',
    log_file: str = None,
):
    """Trains MatryoshkaSAE on user and item embeddings."""
    def _log(msg):
        print(msg, flush=True)
        if log_file:
            with open(log_file, "a", buffering=1) as f:
                f.write(msg + "\n")
                f.flush()

    optimizer = torch.optim.Adam(sae.parameters(), lr=lr)

    for epoch in range(1, num_epochs + 1):
        epoch_loss = 0.0

        for u, i in zip(user_loader, item_loader):
            _, z_u, recons_u = sae(u)
            _, z_i, recons_i = sae(i)

            # 1) Nested reconstruction loss
            recon_u_loss = sum(F.mse_loss(r, u) for r in recons_u)
            recon_i_loss = sum(F.mse_loss(r, i) for r in recons_i)
            recon_term = mse_weight * (recon_u_loss + recon_i_loss)

            # 2) Sparsity penalties
            l1_term = l1_weight * (z_u.abs().mean() + z_i.abs().mean())
            kl_term = kl_weight * (
                kl_divergence_loss(z_u, sparsity_target) +
                kl_divergence_loss(z_i, sparsity_target)
            )

            # 3) Dynamic inner-product loss
            full_u_hat = recons_u[-1]
            full_i_hat = recons_i[-1]
            inner_orig = u @ i.T
            inner_recons = full_u_hat @ full_i_hat.T
            beta = beta_schedule(
                epoch - 1, num_epochs,
                beta_start=inner_start,
                beta_end=inner_end,
                warmup=warmup
            )
            inner_term = beta * F.mse_loss(inner_recons, inner_orig)

            loss = recon_term + l1_term + kl_term + inner_term

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * u.size(0)

        avg_loss = epoch_loss / len(user_loader.dataset)
        _log(f"Epoch {epoch}/{num_epochs} — avg_loss: {avg_loss:.6f}")

    _log("Training complete.")

In [ ]:
# import optuna
# import torch

# --- Replace these with your actual imports/functions ---
# from your_module import train_autoencoder_grad_beta, ms_score_new, df_cosine_sim_matrix

# def objective(trial):
    # 1) Suggest hyperparameters
#     mse_w   = trial.suggest_loguniform('mse_weight', 1e-1, 1e1)    # 0.1 → 10
#     l1_w    = trial.suggest_loguniform('l1_weight', 1e-5, 1e0)    # 1e‑5 → 1
#     kl_w    = trial.suggest_loguniform('kl_weight', 1e-5, 1e-1)   # 1e‑5 → 0.1
#     inner_end = trial.suggest_uniform('beta_end', 0.0, 20.0)       # 0 → 20
#     warmup  = trial.suggest_int('warmup', 5, 15)  # 5 → 15 epochs
    # latent_dim = trial.suggest_int('warmup', 70, 90)  # 70 → 90

    # 2) Train your SAE with this config
#     K = 80 #user_tensor.shape[1]
    # Define nested prefixes: e.g. 25%, 50%, 100% of K
#     prefixes = [K // 4, K // 2, K]
#     sae_shared = MatryoshkaAutoencoder(latent_dim=K, input_dim=K, group_sizes=prefixes,k=10)

#     train_matryoshka(
#         sae_shared,
#         user_loader,
#         item_loader,
#         num_epochs=30,
#         lr=1e-3,
#         mse_weight = mse_w,
#         inner_end= inner_end,
#         warmup     = warmup
#     )

    # 3) Extract the item latents and compute MS Score
    # _,latents_items,_ = sae_shared(dataset_items)  # assume shape [N_items, latent_dim]
#     latents_items,_ = sae_shared(dataset_items)  # assume shape [N_items, latent_dim]
#     avg_ms = ms_score_new(df_cosine_sim_matrix, latents_items)

#     return float(avg_ms)

# 4) Create and run the study
# study = optuna.create_study(
#     direction='maximize',
#     sampler=optuna.samplers.TPESampler(seed=42)
# )
# study.optimize(objective, n_trials=10)

# # 5) Inspect the best trial
# print("Best MS Score:", study.best_value)
# print("Best hyperparameters:")
# for key, val in study.best_params.items():
#     print(f"  {key}: {val}")


# 5a) המר לתוך DataFrame
# df_trials = study.trials_dataframe()

# 5b) הצגה על המסך
# print(df_trials)

# 5c) אופציונלי: מיון לפי ביצועים ושמירה ל־CSV
# df_sorted = df_trials.sort_values("value", ascending=False)
# print("\nTop 5 trials:")
# print(df_sorted.head(5))
# df_sorted.to_csv(OUTPUTS_DIR+'dors/220725_iter_predLoss/optuna_sae_trials.csv', index=False)
# df_sorted.to_csv(OUTPUTS_DIR+'dors/220725_iter_predLoss/optuna_matriyoshka_260725_1.csv', index=False)

# 5d) גרף לדוגמה: MS Score מול beta_end
# plt.figure(figsize=(6,4))
# plt.scatter(df_trials["params_beta_end"], df_trials["value"])
# plt.xlabel("beta_end")
# plt.ylabel("MS Score")
# plt.title("MS Score vs. beta_end")
# plt.show()

# df_trials

In [ ]:
# 5) Inspect the best trial
# print("Best MS Score:", study.best_value)
# print("Best hyperparameters:")
# for key, val in study.best_params.items():
#     print(f"  {key}: {val}")

In [ ]:
# -----------------------------------
# 4) Run training
# -----------------------------------
# K = user_tensor.shape[1]
# Define nested prefixes: e.g. 25%, 50%, 100% of K
# prefixes = [K // 4, K // 2, K]

# sae_shared = MatryoshkaAutoencoder(latent_dim=K, input_dim=K, group_sizes=prefixes, k=10)

# train_matryoshka(
#     sae_shared,
#     user_loader,
#     item_loader,
#     num_epochs=30,
#     lr=1e-3,
#     mse_weight= 3.7183641805732095,
    # l1_weight  = 0.008758419327655994,
    # kl_weight  =  0.050771314168206266,
#     warmup     = 5,
#     inner_end=11.84829137724085,
# )
# train_matryoshka(
#     sae_shared,
#     user_loader,
#     item_loader,
#     num_epochs=30,
#     lr=1e-3
# )

In [ ]:
# model_name ='SAE_NCF_310725_1650_constInner_2_mse0.11714261837903445_l10.0007907172569000147_kl0.9532963530667937_output_13.589849432397903'

In [ ]:
# torch.save(sae_model, Path(OUTPUTS_DIR,f'dors/290725/{model_name}.pth'))

In [ ]:
# ============================================================
# Build dataset_users / dataset_items from trained MF embeddings.
# In the original project these tensors come from NCF embedding
# CSV files (see Copy_of_lastFM_SAE_NCF_111125.ipynb, cell 184).
# Here we use the MF embeddings (mf.P / mf.Q) which are already
# trained above and have the same shape [N, K].
# ============================================================
import torch

dataset_users          = torch.tensor(mf.P, dtype=torch.float32)   # [num_users, K]
dataset_items          = torch.tensor(mf.Q, dtype=torch.float32)   # [num_items, K]
interaction_embeddings = dataset_users                               # all users (no test split here)

print(f"dataset_users          : {dataset_users.shape}")
print(f"dataset_items          : {dataset_items.shape}")
print(f"interaction_embeddings : {interaction_embeddings.shape}")

In [ ]:
# Train Matryoshka SAE on same data as SAE
import os
from torch.utils.data import DataLoader

# ---- logging setup ----
job_id = os.environ.get("SLURM_JOB_ID", "local")
log_file = f"matryoshka_training_{job_id}.txt"

def log(msg):
    print(msg, flush=True)
    with open(log_file, "a", buffering=1) as f:
        f.write(msg + "\n")
        f.flush()

K = dataset_items.shape[1]       # input/output dim = 20
latent_dim_mat = 50              # hidden layer = 50
prefixes = list(range(5, latent_dim_mat + 1, 5))  # [5,10,...,50]

log(f"Job ID: {job_id}")
log(f"dataset_users shape: {dataset_users.shape}")
log(f"dataset_items shape: {dataset_items.shape}")
log(f"K={K}, latent_dim={latent_dim_mat}, prefixes={prefixes}")
log("=" * 50)

# Build Matryoshka model
mat_sae = MatryoshkaAutoencoder(
    latent_dim=latent_dim_mat,
    input_dim=K,
    group_sizes=prefixes,
    activation=torch.nn.ReLU(),
    tied=True,
    normalize=True
)

# DataLoaders
user_loader_mat = DataLoader(dataset_users, batch_size=256, shuffle=True, drop_last=True)
item_loader_mat = DataLoader(dataset_items, batch_size=256, shuffle=True, drop_last=True)

log("Starting Matryoshka SAE training...")

# Train
train_matryoshka(
    mat_sae,
    user_loader_mat,
    item_loader_mat,
    num_epochs=30,
    lr=1e-4,
    mse_weight=3.7183641805732095,
    warmup=10,
    inner_end=5.0,
    log_file=log_file,
)

log("Matryoshka SAE training complete.")

# Save MSAE model
import torch
torch.save(mat_sae.state_dict(), "mat_sae.pth")
print("Saved: mat_sae.pth")

In [ ]:
# ============================================================
# Get MSAE reconstructions for users and items
# (used by evaluation + comparison cells below)
# ============================================================
import torch

with torch.no_grad():
    _, _, recons_u_all = mat_sae(dataset_users)   # list of [N_users, K]
    _, _, recons_i_all = mat_sae(dataset_items)   # list of [N_items, K]

# Use full (last-prefix) reconstruction
recons_users = recons_u_all[-1].numpy()   # [N_users, K]
recons_items = recons_i_all[-1].numpy()   # [N_items, K]

users_eval = np.unique(users_val_pos)     # users that have val positives

print(f"recons_users : {recons_users.shape}")
print(f"recons_items : {recons_items.shape}")
print(f"users_eval   : {len(users_eval)} users")

In [ ]:
import matplotlib
matplotlib.use("Agg")  # non-interactive backend, no display
import matplotlib.pyplot as plt
import numpy as np

# ----------------------------------------------------------
# 1) Get item latents from mat_sae
# ----------------------------------------------------------
with torch.no_grad():
    _, latents_items, _ = mat_sae(dataset_items)

latents_items_np = latents_items.detach().numpy()
num_neurons      = latents_items_np.shape[1]
N_top            = 500

# ----------------------------------------------------------
# 2) Overall mean activation bar chart (saved, not shown)
# ----------------------------------------------------------
mean_activation_per_neuron = latents_items_np.mean(axis=0)
sorted_neuron_idx          = np.argsort(mean_activation_per_neuron)[::-1]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(num_neurons),
       mean_activation_per_neuron[sorted_neuron_idx],
       color='tan', edgecolor='black', width=0.7)
ax.set_xticks(range(num_neurons))
ax.set_xticklabels([f"N{i}" for i in sorted_neuron_idx], fontsize=9)
ax.set_xlabel("Neuron (sorted by mean activation)", fontsize=12, fontweight='bold')
ax.set_ylabel("Mean Activation", fontsize=12, fontweight='bold')
ax.set_title("MSAE — Mean Activation per Neuron (all items)", fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig("msae_overall_neuron_activation.png", dpi=150, bbox_inches='tight')
plt.close()
print("Saved: msae_overall_neuron_activation.png")

In [ ]:
# Save top-500 most activated item indices per neuron to txt file
import numpy as np

job_id = os.environ.get("SLURM_JOB_ID", "local")
out_path = f"msae_top500_items_per_neuron_{job_id}.txt"

lines = []
for neuron_idx in range(num_neurons):
    activations = latents_items_np[:, neuron_idx]
    top_500 = np.argsort(activations)[-500:][::-1].tolist()
    lines.append("N" + str(neuron_idx) + ": " + str(top_500))

with open(out_path, "w") as f:
    f.writelines(line + chr(10) for line in lines)

print(f"Saved: {out_path}")

In [ ]:
print(K)

#SAE training and comaprring

In [ ]:
# ============================================================
# Train plain SAE (adapted from lastFM_SAE_NCF_with_MSAE.ipynb)
# Uses MF inner product as output loss instead of NCF
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import itertools, os
import numpy as np

job_id  = os.environ.get("SLURM_JOB_ID", "local")
sae_log = f"sae_training_{job_id}.txt"

def slog(msg):
    print(msg, flush=True)
    with open(sae_log, "a", buffering=1) as f:
        f.write(msg + "\n")
        f.flush()

# ----------------------------------------------------------
# SAE model (from lastFM_SAE_NCF_with_MSAE, adapted)
# ----------------------------------------------------------
class SparseAutoencoderMF(nn.Module):
    def __init__(self, input_dim, hidden_dim, tie_weights=True):
        super().__init__()
        self.encoder_linear = nn.Linear(input_dim, hidden_dim, bias=False)
# self.encoder_linear = nn.Linear(input_dim, hidden_dim)
        self.activation     = nn.ReLU()
        if tie_weights:
            self.decoder = TiedTranspose(self.encoder_linear)
        else:
            self.decoder = nn.Linear(hidden_dim, input_dim, bias=False)
        self.loss = []

    def forward(self, x):
        latent_pre_act = self.encoder_linear(x)
        encoded        = self.activation(latent_pre_act)
        decoded        = self.decoder(encoded)
        return decoded, encoded


# ----------------------------------------------------------
# SAE training function (MF output loss instead of NCF)
# ----------------------------------------------------------
def train_sae_mf(
    sae, dataset_users, dataset_items,
    mf_b_u, mf_b_i,
    epochs=30, lr=1e-3, batch_size=256,
    mse_weight=2.6, sparsity_weight=1e-5,
    kl_weight=0.7, output_loss_weight=10.0,
    sparsity_target=0.05, log_file=None
):
    """
    Train a plain SAE on MF embeddings.
    Output loss = MSE between inner products:
        (b_u + b_i + P @ Q.T)  vs  (b_u + b_i + P_rec @ Q_rec.T)
    """
    def _log(msg):
        print(msg, flush=True)
        if log_file:
            with open(log_file, "a", buffering=1) as f:
                f.write(msg + "\n")
                f.flush()

    optimizer   = optim.Adam(sae.parameters(), lr=lr)
    mse_loss_fn = nn.MSELoss()

    b_u = torch.tensor(mf_b_u, dtype=torch.float32)
    b_i = torch.tensor(mf_b_i, dtype=torch.float32)

    g_u = torch.Generator().manual_seed(42)
    g_i = torch.Generator().manual_seed(42)

    user_indices = torch.arange(dataset_users.shape[0])
    item_indices = torch.arange(dataset_items.shape[0])

    dl_users = DataLoader(user_indices, batch_size=batch_size,
                          shuffle=True, drop_last=True, generator=g_u)
    dl_items = DataLoader(item_indices, batch_size=batch_size,
                          shuffle=True, drop_last=True, generator=g_i)

    for epoch in range(1, epochs + 1):
        epoch_loss   = 0.0
        samples_seen = 0

        for u_idx, i_idx in zip(dl_users, itertools.cycle(dl_items)):
            u_idx = u_idx.long()
            i_idx = i_idx.long()

            user_emb = dataset_users[u_idx]
            item_emb = dataset_items[i_idx]

            user_rec, user_enc = sae(user_emb)
            item_rec, item_enc = sae(item_emb)

            loss_recon = (mse_loss_fn(user_rec, user_emb) +
                          mse_loss_fn(item_rec, item_emb))

            loss_l1 = (user_enc.abs().mean() +
                       item_enc.abs().mean())

            loss_kl = (kl_divergence_loss(user_enc, sparsity_target) +
                       kl_divergence_loss(item_enc, sparsity_target))

            b_u_batch = b_u[u_idx].unsqueeze(1).expand(-1, len(i_idx))
            b_i_batch = b_i[i_idx].unsqueeze(0).expand(len(u_idx), -1)

            inner_orig  = b_u_batch + b_i_batch + user_emb @ item_emb.T
            inner_recon = b_u_batch + b_i_batch + user_rec @ item_rec.T
            loss_output = mse_loss_fn(inner_recon, inner_orig)

            loss = (mse_weight         * loss_recon  +
                    sparsity_weight    * loss_l1     +
                    kl_weight          * loss_kl     +
                    output_loss_weight * loss_output)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batch_size_actual = user_emb.size(0)
            epoch_loss   += loss.item() * batch_size_actual
            samples_seen += batch_size_actual

        avg = epoch_loss / samples_seen
        sae.loss.append(avg)
        _log(f"SAE Epoch {epoch}/{epochs} — avg_loss: {avg:.6f}")

    _log("SAE training complete.")


# ----------------------------------------------------------
# Instantiate and train
# ----------------------------------------------------------
K_sae      = dataset_items.shape[1]
hidden_dim = K_sae

sae_model = SparseAutoencoderMF(
    input_dim=K_sae,
    hidden_dim=hidden_dim,
    tie_weights=True
)

slog(f"Training plain SAE — input_dim={K_sae}, hidden_dim={hidden_dim}")

train_sae_mf(
    sae_model,
    dataset_users,
    dataset_items,
    mf_b_u=mf.b_u,
    mf_b_i=mf.b_i,
    epochs=30,
    lr=1e-4,
    batch_size=256,
    mse_weight=2.6,
    sparsity_weight=1e-5,
    kl_weight=0.7,
    output_loss_weight=10.0,
    log_file=sae_log,
)

# Get SAE reconstructions for evaluation
with torch.no_grad():
    recons_u_sae_, enc_u_sae_ = sae_model(dataset_users)
    recons_i_sae_, enc_i_sae_ = sae_model(dataset_items)

recons_users_sae = recons_u_sae_.numpy()
recons_items_sae = recons_i_sae_.numpy()

slog(f"SAE reconstructions ready: users {recons_users_sae.shape}, items {recons_items_sae.shape}")
# Save SAE model
torch.save(sae_model.state_dict(), "sae_model.pth")
print("Saved: sae_model.pth")

In [ ]:
import matplotlib
matplotlib.use("Agg")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, log_loss

K_rec = 10

def recommend_sae(user_idx, top_k=K_rec):
    scores = (mf.b_u[user_idx] + mf.b_i + recons_users_sae[user_idx] @ recons_items_sae.T)
    seen = set(R_train[user_idx].indices)
    scores[list(seen)] = -np.inf
    return np.argsort(scores)[-top_k:][::-1].tolist()

def recommend_msae(user_idx, top_k=K_rec):
    scores = (mf.b_u[user_idx] + mf.b_i + recons_users[user_idx] @ recons_items.T)
    seen = set(R_train[user_idx].indices)
    scores[list(seen)] = -np.inf
    return np.argsort(scores)[-top_k:][::-1].tolist()

def eval_metrics(rec_fn, label):
    precisions, recalls, ndcgs = [], [], []
    for u in users_eval:
        true_items = set(R_val[u].indices)
        if not true_items:
            continue
        recs = rec_fn(u, top_k=K_rec)
        hits = len(set(recs) & true_items)
        precisions.append(hits / K_rec)
        recalls.append(hits / len(true_items))
        dcg  = sum(1 / np.log2(i + 2) for i, item in enumerate(recs) if item in true_items)
        idcg = sum(1 / np.log2(i + 2) for i in range(min(len(true_items), K_rec)))
        ndcgs.append(dcg / idcg if idcg > 0 else 0)
    p = float(np.mean(precisions))
    r = float(np.mean(recalls))
    n = float(np.mean(ndcgs))
    slog(f"{label:10s}  Precision@{K_rec}: {p:.4f}  Recall@{K_rec}: {r:.4f}  NDCG@{K_rec}: {n:.4f}")
    return p, r, n

def eval_auc(recons_u, recons_i, label):
    scores = (mf.b_u[users_val] + mf.b_i[items_val] + np.sum(recons_u[users_val] * recons_i[items_val], axis=1))
    auc = roc_auc_score(y_val, scores)
    ll  = log_loss(y_val, 1 / (1 + np.exp(-scores)))
    acc = ((scores > 0).astype(int) == y_val).mean()
    slog(f"{label:10s}  AUC: {auc:.4f}  LogLoss: {ll:.4f}  Acc: {acc:.4f}")
    return auc, ll, acc

slog("\n--- SAE vs MSAE Evaluation ---")
auc_sae,  ll_sae,  acc_sae  = eval_auc(recons_users_sae, recons_items_sae, "SAE")
auc_msae, ll_msae, acc_msae = eval_auc(recons_users,     recons_items,     "MSAE")

slog("\n--- Ranking Metrics ---")
p_sae,  r_sae,  n_sae  = eval_metrics(recommend_sae,  "SAE")
p_msae, r_msae, n_msae = eval_metrics(recommend_msae, "MSAE")

np.random.seed(42)
sample_users = np.random.choice(users_eval, size=min(20, len(users_eval)), replace=False)
rows = []
for u in sample_users:
    recs_sae  = recommend_sae(u,  top_k=K_rec)
    recs_msae = recommend_msae(u, top_k=K_rec)
    overlap     = set(recs_sae) & set(recs_msae)
    overlap_pct = len(overlap) / K_rec * 100
    rank_sae  = {item: r for r, item in enumerate(recs_sae)}
    rank_msae = {item: r for r, item in enumerate(recs_msae)}
    common    = set(recs_sae) & set(recs_msae)
    avg_shift = np.mean([abs(rank_sae[item] - rank_msae[item]) for item in common]) if common else K_rec
    rows.append({"user": u, "overlap_%": round(overlap_pct, 1), "avg_rank_shift": round(avg_shift, 2)})

df_compare = pd.DataFrame(rows)

slog("\n" + "=" * 60)
slog(f"{'Metric':<20} {'SAE':>10} {'MSAE':>10} {'Diff':>10}")
slog("-" * 60)
slog(f"{'AUC':<20} {auc_sae:>10.4f} {auc_msae:>10.4f} {auc_msae-auc_sae:>+10.4f}")
slog(f"{'Log Loss':<20} {ll_sae:>10.4f} {ll_msae:>10.4f} {ll_msae-ll_sae:>+10.4f}")
slog(f"{'Accuracy':<20} {acc_sae:>10.4f} {acc_msae:>10.4f} {acc_msae-acc_sae:>+10.4f}")
slog(f"{'Precision@10':<20} {p_sae:>10.4f} {p_msae:>10.4f} {p_msae-p_sae:>+10.4f}")
slog(f"{'Recall@10':<20} {r_sae:>10.4f} {r_msae:>10.4f} {r_msae-r_sae:>+10.4f}")
slog(f"{'NDCG@10':<20} {n_sae:>10.4f} {n_msae:>10.4f} {n_msae-n_sae:>+10.4f}")
slog(f"{'Avg overlap %':<20} {df_compare['overlap_%'].mean():>10.1f}%")
slog(f"{'Avg rank shift':<20} {df_compare['avg_rank_shift'].mean():>10.2f} pos")
slog("=" * 60)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
metrics   = ["AUC", "Precision@10", "Recall@10", "NDCG@10"]
sae_vals  = [auc_sae,  p_sae,  r_sae,  n_sae]
msae_vals = [auc_msae, p_msae, r_msae, n_msae]
x = np.arange(len(metrics))
axes[0].bar(x - 0.2, sae_vals,  0.4, label="SAE",  color="steelblue")
axes[0].bar(x + 0.2, msae_vals, 0.4, label="MSAE", color="coral")
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics, rotation=15)
axes[0].set_title("SAE vs MSAE — Metrics")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)
axes[1].hist(df_compare["overlap_%"], bins=10, color="steelblue", edgecolor="black")
axes[1].axvline(df_compare["overlap_%"].mean(), color="red", linestyle="--",
                label=f"Mean: {df_compare['overlap_%'].mean():.1f}%")
axes[1].set_xlabel("Overlap % (SAE vs MSAE top-10)")
axes[1].set_ylabel("Number of users")
axes[1].set_title("Recommendation overlap")
axes[1].legend()
axes[2].hist(df_compare["avg_rank_shift"], bins=10, color="coral", edgecolor="black")
axes[2].axvline(df_compare["avg_rank_shift"].mean(), color="red", linestyle="--",
                label=f"Mean: {df_compare['avg_rank_shift'].mean():.2f}")
axes[2].set_xlabel("Avg rank shift (positions)")
axes[2].set_ylabel("Number of users")
axes[2].set_title("Rank shift: SAE vs MSAE")
axes[2].legend()
plt.tight_layout()
plt.savefig("sae_vs_msae_comparison.png", dpi=150, bbox_inches="tight")
plt.close()
slog("Saved: sae_vs_msae_comparison.png")

# -----------------------------------------------

# Imports

In [218]:
# ===== Imports =====
import os, json, pickle
import numpy as np
import pandas as pd
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")

from collections import Counter

from scipy.special import expit
from scipy.stats import entropy as scipy_entropy
from scipy.spatial.distance import jensenshannon

from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
import random
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

## Load MF pickle

In [219]:
# ===== Load MF model =====
class SparseMF:
    pass

with open("mf_model.pkl", "rb") as f:
    mf = pickle.load(f)

print("Loaded MF.")
print("P:", mf.P.shape, "Q:", mf.Q.shape, "K:", mf.K)

Loaded MF.
P: (400000, 20) Q: (182207, 20) K: 20


## Rebuild MatryoshkaAutoencoder exactly like training

In [220]:
# ===== Load MSAE (MatryoshkaAutoencoder) =====
device = "cuda" if torch.cuda.is_available() else "cpu"

K = mf.Q.shape[1]            # 20
latent_dim_mat = 50          # 50 neurons
prefixes = list(range(5, latent_dim_mat + 1, 5))

mat_sae = MatryoshkaAutoencoder(
    latent_dim=latent_dim_mat,
    input_dim=K,
    group_sizes=prefixes,
    activation=torch.nn.ReLU(),
    tied=True,
    normalize=True
).to(device)

state = torch.load("mat_sae.pth", map_location=device)

# Remove extra metadata keys that your saved state_dict included
state.pop("activation", None)
state.pop("activation_state_dict", None)

mat_sae.load_state_dict(state, strict=True)
mat_sae.eval()
print("Loaded MSAE:", latent_dim_mat, "neurons")

Loaded MSAE: 50 neurons


## Build tensors for embeddings

In [221]:
P = mf.P.astype(np.float32)  # users
Q = mf.Q.astype(np.float32)  # items
b_u = mf.b_u.astype(np.float32)
b_i = mf.b_i.astype(np.float32)

dataset_users = torch.tensor(P, dtype=torch.float32, device=device)
dataset_items = torch.tensor(Q, dtype=torch.float32, device=device)

## Load neuron labels (refined labels)

In [222]:
# ===== Load neuron labels =====
labels_path = "all_neuron_labels.json"
with open(labels_path, "r", encoding="utf-8") as f:
    neuron_labels = json.load(f)

def refined_suffix(s: str) -> str:
    """
    take refined_label and keep only the part AFTER the em dash (—) only if broad_label is duplicated
     """
    if not isinstance(s, str):
        return ""
    return s.split("—", 1)[1].strip() if "—" in s else s.strip()

# count broad label duplicates
broad_counts = Counter(d.get("broad_label", "").strip() for d in neuron_labels if d.get("broad_label"))

# neuron_id -> axis label 
neuron_id_to_axis_label = {}
for d in neuron_labels:
    nid = int(d["neuron_id"])
    broad = (d.get("broad_label") or "").strip()
    refined = refined_suffix(d.get("refined_label", ""))

    if broad and broad_counts[broad] == 1:
        axis_lbl = broad                          # unique broad label -> use broad
    else:
        axis_lbl = refined or broad or ""          # duplicated broad -> use refined suffix

    neuron_id_to_axis_label[nid] = axis_lbl

In [223]:
neuron_id_to_axis_label

{0: 'general mixed apparel and accessories',
 1: 'mixed family apparel & footwear',
 2: 'mixed gender apparel & footwear',
 3: "women's apparel & accessories dominant",
 4: 'mixed gender apparel & accessories',
 5: 'socks, underwear & basics',
 6: 'mixed accessories & apparel basics',
 7: "mixed women's apparel & accessories",
 8: 'mixed gender apparel & footwear',
 9: 'mixed apparel, accessories & jewelry',
 10: 'general mixed apparel and accessories',
 11: 'mixed apparel and accessories across demographics',
 12: 'mixed family apparel & accessories',
 13: "mixed apparel and accessories (women's dresses, men's activewear, footwear)",
 14: 'mixed gender footwear & apparel',
 15: 'mixed gender apparel & accessories',
 16: 'mixed gender apparel & accessories',
 17: 'mixed gender apparel & footwear',
 18: "mixed women's/family footwear & apparel",
 19: 'mixed gender apparel & accessories',
 20: 'silicone/rubber rings & general apparel mix',
 21: 'mixed general apparel & accessories',
 22:

### Create a labeled mean activation histogram like before

In [224]:
# 1) Get item latents from mat_sae
with torch.no_grad():
    _, latents_items, _ = mat_sae(dataset_items)

latents_items_np = latents_items.detach().cpu().numpy()
num_neurons = latents_items_np.shape[1]

# 2) Mean activation
mean_activation_per_neuron = latents_items_np.mean(axis=0)
sorted_neuron_idx = np.argsort(mean_activation_per_neuron)[::-1]

# 3) Plot
fig, ax = plt.subplots(figsize=(22, 7))  
ax.bar(
    range(num_neurons),
    mean_activation_per_neuron[sorted_neuron_idx],
    color="tan", edgecolor="black", width=0.7
)

ax.set_xticks(range(num_neurons))

# 4) Label each tick with neuron id + rule based label
xt = []
for i in sorted_neuron_idx:
    lbl = neuron_id_to_axis_label.get(int(i), "")
    xt.append(f"N{i}: {lbl}" if lbl else f"N{i}")

ax.set_xticklabels(xt, fontsize=9, rotation=45, ha="right")

ax.set_xlabel("Neuron (sorted by mean activation)", fontsize=12, fontweight="bold")
ax.set_ylabel("Mean Activation", fontsize=12, fontweight="bold")
ax.set_title("MSAE — Mean Activation per Neuron (all items)", fontsize=14, fontweight="bold")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("msae_overall_neuron_activation_labeled.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: msae_overall_neuron_activation_labeled.png")

# 5) Save a table of the activation as well
df = pd.DataFrame({
    "neuron_id": sorted_neuron_idx,
    "mean_activation": mean_activation_per_neuron[sorted_neuron_idx],
    "axis_label": [neuron_id_to_axis_label.get(int(i), "") for i in sorted_neuron_idx]
})
df.to_csv("msae_neuron_mean_activation_with_labels.csv", index=False)
print("Saved: msae_neuron_mean_activation_with_labels.csv")

Saved: msae_overall_neuron_activation_labeled.png
Saved: msae_neuron_mean_activation_with_labels.csv


## Story - Gender Axis Intervention 

### Item gender tag function (Women% / Men%)

In [225]:
# ===== Load item metadata for tagging =====
items_df = pd.read_json("items_metadata.jsonl", lines=True).reset_index(drop=True)
items_df = items_df.iloc[:Q.shape[0]].copy()

In [226]:
def item_gender_tag(item_idx: int) -> str:
    """
    Determine gender tag of an item based on its category list.

    Returns
    -------
    "women" | "men" | "neutral"
    """

    cats = items_df.loc[int(item_idx), "categories"]

    if not isinstance(cats, list):
        return "neutral"

    # normalize categories
    cats_norm = {str(c).strip().lower() for c in cats}

    if "women" in cats_norm:
        return "women"

    if "men" in cats_norm:
        return "men"

    return "neutral"

### Recommendation function (baseline)

In [227]:
def recommend_topk_from_item_emb(P_user: np.ndarray, Q_items: np.ndarray, b_u: np.ndarray, b_i: np.ndarray,
                                user_idx: int, K: int = 50, seen: set[int] | None = None):
    scores = b_u[user_idx] + b_i + (P_user[user_idx] @ Q_items.T)  # [n_items]
    if seen is not None and len(seen) > 0:
        scores[list(seen)] = -np.inf
    top = np.argpartition(-scores, K)[:K]
    top = top[np.argsort(-scores[top])]
    return top, scores[top]

### Get latent Z for items + scale specific neurons

In [228]:
@torch.no_grad()
def msae_reconstruct_with_scaled_neurons(mat_sae: MatryoshkaAutoencoder,
                                        X: torch.Tensor,
                                        neuron_ids: list[int],
                                        scale: float):
    """
    X: [N, K] embeddings
    neuron_ids: which latent dims to scale multiplicatively
    scale: >1 amplify, <1 suppress (e.g. 1.5 or 0.5)
    Returns: X_hat [N, K], Z [N, 50]
    """
    X_norm, info = mat_sae.preprocess(X)
    z_pre = mat_sae.encode_pre_act(X_norm)
    z = mat_sae.activation(z_pre)  # [N, 50]

    z2 = z.clone()
    for nid in neuron_ids:
        z2[:, nid] = z2[:, nid] * scale

    # decode full latent (no prefix masking)
    X_hat = mat_sae.decode(z2, info)
    return X_hat, z2

In [229]:
# Story Neurons from LLM
WOMEN_DOM_NEURONS = [3, 32, 39, 41, 45, 48]

#### Neuron Wise Diagnostic

Neuron wise diagnostic plot - for the causal interpertation we want to know if the neurons we identified actually control the gender shift by checking how much each neuron individually shifts women_ratio
For each neuron k:
1. Reconstruct item embeddings with that neuron amplified
2. Generate recommendations
3. Measure change in women_ratio
4. Plot the delta in women ratio
This way we know if neuron activation actually changes recommendation distribution

In [ ]:
"""
def neuron_gender_effect_diagnostic(
    users,
    P, Q, b_u, b_i,
    mat_sae,
    dataset_items,
    items_df,
    item_gender_tag_fn,
    recommend_topk_from_item_emb_fn,
    msae_reconstruct_with_scaled_neurons_fn,
    R_train,
    topK=50,
    scale=2.0
):

    with torch.no_grad():
        _, latents_tmp, _ = mat_sae(dataset_items[:1])

    num_neurons = latents_tmp.shape[1]

    effects = []

    for neuron in range(num_neurons):

        # amplify only this neuron and reconstruct only once
        Q_hat,_ = msae_reconstruct_with_scaled_neurons_fn(
            mat_sae,
            dataset_items,
            [neuron],
            scale
        )

        Q_new = Q_hat.detach().cpu().numpy()

        deltas = []

        for u in users:

            seen = set(R_train[u].indices)

            # baseline
            base_top,_ = recommend_topk_from_item_emb_fn(
                P,Q,b_u,b_i,u,K=topK,seen=seen
            )

            base_gender = gender_metrics(
                base_top,
                item_gender_tag_fn=item_gender_tag_fn
            )

            new_top,_ = recommend_topk_from_item_emb_fn(
                P,Q_new,b_u,b_i,u,K=topK,seen=seen
            )

            new_gender = gender_metrics(
                new_top,
                item_gender_tag_fn=item_gender_tag_fn
            )

            deltas.append(
                new_gender["women_to_men_ratio"] -
                base_gender["women_to_men_ratio"]
            )

        effects.append(np.mean(deltas))

    return np.array(effects)
"""

##### Plotting the Results

In [ ]:
"""
np.random.seed(42)

diagnostic_users = np.random.choice(
    np.arange(P.shape[0]),
    size=500,
    replace=False
)

effects = neuron_gender_effect_diagnostic(
    users=diagnostic_users,
    P=P,Q=Q,b_u=b_u,b_i=b_i,
    mat_sae=mat_sae,
    dataset_items=dataset_items,
    items_df=items_df,
    item_gender_tag_fn=item_gender_tag,
    recommend_topk_from_item_emb_fn=recommend_topk_from_item_emb,
    msae_reconstruct_with_scaled_neurons_fn=msae_reconstruct_with_scaled_neurons,
    R_train=R_train
)

top = np.argsort(effects)[::-1][:10]

print("Top positive neurons:")
for n in top:
    print(f"N{n}: Δratio = {effects[n]:.4f}")

bottom = np.argsort(effects)[:10]

print("\nTop negative neurons:")
for n in bottom:
    print(f"N{n}: Δratio = {effects[n]:.4f}")
    
plt.figure(figsize=(12,4))

plt.bar(range(len(effects)),effects)
plt.axhline(0,color="black")

plt.xlabel("Neuron ID")
plt.ylabel("Δ Women/Men ratio")

plt.title("Neuron influence on gender bias in recommendations")

plt.grid(alpha=0.3)
plt.savefig("neuron_gender_effect.png", dpi=200, bbox_inches="tight")
plt.close()

print("Saved: neuron_gender_effect.png")
"""

Top positive neurons:
N10: Δratio = 0.0299
N31: Δratio = 0.0235
N2: Δratio = 0.0217
N16: Δratio = 0.0215
N28: Δratio = 0.0194
N33: Δratio = 0.0185
N17: Δratio = 0.0184
N26: Δratio = 0.0171
N36: Δratio = 0.0161
N19: Δratio = 0.0157

Top negative neurons:
N22: Δratio = -0.0013
N9: Δratio = 0.0013
N29: Δratio = 0.0035
N23: Δratio = 0.0041
N8: Δratio = 0.0054
N4: Δratio = 0.0073
N13: Δratio = 0.0078
N11: Δratio = 0.0078
N20: Δratio = 0.0078
N14: Δratio = 0.0085
Saved: neuron_gender_effect.png


In [ ]:
"""
WOMEN_DOM_NEURONS_FROM_LLM = [3, 32, 39, 41, 45, 48]
WOMEN_DOM_NEURONS_DIAGNOSTIC = [10, 31, 2, 16]
WOMEN_DOM_NEURONS = WOMEN_DOM_NEURONS_DIAGNOSTIC # No overlap!
"""

### Compute Monosemanticity Score (MS)

In [230]:
# -------------------------------------------------
# Extract SAE latent activations for all items
# -------------------------------------------------

with torch.no_grad():
    _, latents_items, _ = mat_sae(dataset_items)

latents_items = latents_items.detach().cpu().numpy()

In [231]:
# -------------------------------------------------
# Subsample items for MS computation as computing pairwise similarity for all items is impossible
# -------------------------------------------------
np.random.seed(42)

subset_size = 5000

subset_idx = np.random.choice(
    latents_items.shape[0],
    size=subset_size,
    replace=False
)

latents_sub = latents_items[subset_idx]
items_sub   = dataset_items[subset_idx].detach().cpu().numpy()

print("Subset size:", subset_size)

Subset size: 5000


In [232]:
# -------------------------------------------------
# Compute pairwise item similarity
# -------------------------------------------------

print("Computing cosine similarity...")

S = cosine_similarity(items_sub)

print("Similarity matrix shape:", S.shape)

Computing cosine similarity...
Similarity matrix shape: (5000, 5000)


In [233]:
# -------------------------------------------------
# Min-max normalize neuron activations - Eq.7 in the paper
# -------------------------------------------------

def normalize_activations(A):

    A_min = A.min(axis=0, keepdims=True)
    A_max = A.max(axis=0, keepdims=True)

    return (A - A_min) / (A_max - A_min + 1e-9)

A_norm = normalize_activations(latents_sub)

print("Normalized activation shape:", A_norm.shape)

Normalized activation shape: (5000, 50)


In [234]:
# -------------------------------------------------
# Compute Monosemanticity Score (MS) - Equations (8) and (9) in the paper 
# -------------------------------------------------

def compute_monosemanticity_scores(A_norm, similarity_matrix):

    N, num_neurons = A_norm.shape

    ms_scores = []

    for k in range(num_neurons):

        a = A_norm[:, k]

        # relevance matrix
        relevance = np.outer(a, a)

        # ignore identical pairs
        np.fill_diagonal(relevance, 0)

        numerator   = np.sum(relevance * similarity_matrix)
        denominator = np.sum(relevance) + 1e-9

        score = numerator / denominator

        ms_scores.append(score)

    return np.array(ms_scores)


ms_scores = compute_monosemanticity_scores(A_norm, S)

print("Computed MS for", len(ms_scores), "neurons")

Computed MS for 50 neurons


In [235]:
# -------------------------------------------------
# Save MS scores
# -------------------------------------------------

df_ms = pd.DataFrame({
    "neuron_id": np.arange(len(ms_scores)),
    "monosemanticity_score": ms_scores
})

df_ms.to_csv("msae_monosemanticity_scores.csv", index=False)

print(df_ms.head())

   neuron_id  monosemanticity_score
0          0               0.078916
1          1               0.077215
2          2               0.082632
3          3               0.076034
4          4               0.094011


In [236]:
# -------------------------------------------------
# Plot MS score distribution
# -------------------------------------------------

plt.figure(figsize=(8,5))

sns.histplot(
    ms_scores,
    bins=15,
    kde=True,
    edgecolor="black"
)

plt.xlabel("Monosemanticity Score")
plt.ylabel("Number of Neurons")

plt.title("Distribution of Monosemanticity Scores Across MSAE Neurons")

plt.grid(alpha=0.3)

plt.savefig(
    "msae_ms_score_distribution_kde.png",
    dpi=250,
    bbox_inches="tight"
)

plt.close()

In [237]:
# -------------------------------------------------
# MS per neuron
# -------------------------------------------------

sorted_idx = np.argsort(ms_scores)[::-1]

plt.figure(figsize=(12,4))

plt.bar(
    range(len(ms_scores)),
    ms_scores[sorted_idx],
    edgecolor="black"
)

plt.xticks(
    range(len(ms_scores)),
    sorted_idx,
    rotation=60
)

plt.xlabel("Neuron ID (sorted by MS)")
plt.ylabel("Monosemanticity Score")

plt.title("Monosemanticity Scores per MSAE Neuron")

plt.grid(alpha=0.3)

plt.savefig(
    "msae_ms_score_per_neuron_sorted.png",
    dpi=250,
    bbox_inches="tight"
)

plt.close()

In [ ]:
"""
# -------------------------------------------------
# MS vs Causal Effect
# -------------------------------------------------

plt.figure(figsize=(7,7))

sns.scatterplot(x=ms_scores,
                y=effects,
                hue=["Selected neuron" if i in WOMEN_DOM_NEURONS else "Other neuron" for i in range(len(ms_scores))],
                palette={"Selected neuron":"red","Other neuron":"steelblue"},
                s=80)

# annotate neuron IDs
for i, (x, y) in enumerate(zip(ms_scores, effects)):
    plt.text(x+0.001, y, str(i), fontsize=8)

plt.xlabel("Monosemanticity Score")
plt.ylabel("Δ Women/Men Recommendation Ratio")

plt.title("Relationship Between Neuron Monosemanticity and Gender Causal Effect")

plt.axhline(0, color="black", linewidth=1)

plt.grid(alpha=0.3)

plt.savefig(
    "msae_ms_vs_gender_effect.png",
    dpi=250,
    bbox_inches="tight"
)

plt.close()
"""

In [238]:
# -------------------------------------------------
# Top interpretable neurons
# -------------------------------------------------

top = np.argsort(ms_scores)[::-1][:20]

print("Top MS neurons:")

for n in top:
    print(f"N{n}: MS = {ms_scores[n]:.4f}")

Top MS neurons:
N41: MS = 0.2010
N44: MS = 0.1956
N43: MS = 0.1920
N40: MS = 0.1882
N36: MS = 0.1826
N46: MS = 0.1812
N39: MS = 0.1795
N47: MS = 0.1778
N45: MS = 0.1767
N42: MS = 0.1766
N37: MS = 0.1750
N38: MS = 0.1725
N34: MS = 0.1702
N31: MS = 0.1682
N48: MS = 0.1673
N30: MS = 0.1662
N35: MS = 0.1650
N49: MS = 0.1636
N32: MS = 0.1617
N29: MS = 0.1536


### Metric Functions

In [239]:
def gender_metrics(top_items: np.ndarray, item_gender_tag_fn):
    """
    Compute gender composition metrics for a recommendation list.

    Parameters
    ----------
    top_items : np.ndarray
        Array of item indices (e.g., top-K recommended item ids).
    item_gender_tag_fn : callable
        Function item_gender_tag_fn(item_id:int) -> {"women","men","neutral"} (or similar).
        You already have item_gender_tag(...). Pass it in so this function stays reusable.

    Returns
    -------
    dict
        women_pct : float
            Fraction of items tagged as "women" in the top-K list.
        men_pct : float
            Fraction of items tagged as "men" in the top-K list.
        neutral_pct : float
            Fraction of items tagged as "neutral" (or other) in the top-K list.
        women_to_men_ratio : float
            women_pct / men_pct (with tiny epsilon for stability).
            Interpretable as: how much more “women” than “men” is the list.
    """
    tags = [item_gender_tag_fn(int(i)) for i in top_items]

    w = np.mean([t == "women" for t in tags]) if len(tags) else 0.0
    m = np.mean([t == "men" for t in tags]) if len(tags) else 0.0
    n = np.mean([t == "neutral" for t in tags]) if len(tags) else 0.0

    ratio = (w + 1e-9) / (m + 1e-9)

    return {
        "women_pct": float(w),
        "men_pct": float(m),
        "neutral_pct": float(n),
        "women_to_men_ratio": float(ratio),
    }

In [240]:
def overlap_at_k(a: np.ndarray, b: np.ndarray):
    """
    Measure how similar two top-K recommendation sets are (set overlap).

    Parameters
    ----------
    a, b : np.ndarray
        Two ranked lists (or just arrays) of item indices. Typically both length K.

    Returns
    -------
    float
        Overlap fraction in [0,1]:
        |A ∩ B| / |A|   (same as recall of A under B, if |A|=K).
        1.0 means identical sets, 0.0 means no shared items.
    """
    A, B = set(map(int, a)), set(map(int, b))
    return len(A & B) / max(1, len(A))

In [241]:
def rank_shift(before: np.ndarray, after: np.ndarray):
    """
    Quantify how much the *ordering* changed between two ranked top-K lists.

    We compute the average absolute rank difference over items that appear in BOTH lists.
    If there are no common items, returns NaN.

    Parameters
    ----------
    before, after : np.ndarray
        Ranked arrays of item indices (top-K).

    Returns
    -------
    float
        Mean absolute rank shift (in positions).
        Example: 2.4 means shared items moved ~2.4 positions on average.
    """
    rb = {int(it): r for r, it in enumerate(before)}
    ra = {int(it): r for r, it in enumerate(after)}
    common = set(rb) & set(ra)
    if not common:
        return float("nan")
    return float(np.mean([abs(rb[i] - ra[i]) for i in common]))

In [242]:
def category_token_counts(top_items: np.ndarray, items_df: pd.DataFrame, categories_col: str = "categories"):
    """
    Count category token frequency across the recommended items.

    This treats each category token as a 'word' and counts how often it appears
    across the K items. If item i has 5 categories, it contributes +1 to each token.

    Parameters
    ----------
    top_items : np.ndarray
        Array of item indices (top-K items).
    items_df : pd.DataFrame
        Must contain categories_col; items_df.loc[item_id, categories_col] returns list (or NaN).
    categories_col : str
        Column containing list of category tokens.

    Returns
    -------
    dict[str, int]
        counts[token] = how many of the top-K items contain this token.
    """
    counts = {}
    for i in top_items:
        cats = items_df.loc[int(i), categories_col]
        if not isinstance(cats, list):
            continue
        for c in cats:
            c = str(c)
            counts[c] = counts.get(c, 0) + 1
    return counts

In [243]:
def category_token_counts(top_items: np.ndarray, items_df: pd.DataFrame, categories_col: str = "categories"):
    """
    Count category token frequency across the recommended items.

    This treats each category token as a 'word' and counts how often it appears
    across the K items. If item i has 5 categories, it contributes +1 to each token.

    Parameters
    ----------
    top_items : np.ndarray
        Array of item indices (top-K items).
    items_df : pd.DataFrame
        Must contain categories_col; items_df.loc[item_id, categories_col] returns list (or NaN).
    categories_col : str
        Column containing list of category tokens.

    Returns
    -------
    dict[str, int]
        counts[token] = how many of the top-K items contain this token.
    """
    counts = {}
    for i in top_items:
        cats = items_df.loc[int(i), categories_col]
        if not isinstance(cats, list):
            continue
        for c in cats:
            c = str(c)
            counts[c] = counts.get(c, 0) + 1
    return counts

In [244]:
def counts_to_prob(counts: dict):
    """
    Convert a token->count dictionary to a probability distribution.

    Returns keys and probabilities aligned by sorted(keys).

    Returns
    -------
    (keys, p)
      keys : list[str]
      p : np.ndarray
          probability vector summing to 1
    """
    keys = sorted(counts.keys())
    vec = np.array([counts[k] for k in keys], dtype=np.float64)
    p = vec / (vec.sum() + 1e-12)
    return keys, p

In [245]:
def kl_divergence_from_baseline(counts_base: dict, counts_new: dict):
    """
    Compute KL divergence between category distributions:
        KL(p_new || p_base)

    Interpretation
    --------------
    - 0 means the new category distribution matches baseline exactly.
    - Larger means bigger *distributional shift* in categories.

    Note: KL is asymmetric (KL(p_new||p_base) != KL(p_base||p_new)).

    Returns
    -------
    float
        KL divergence (nats).
    """
    keys = sorted(set(counts_base) | set(counts_new))
    pb = np.array([counts_base.get(k, 0) for k in keys], dtype=np.float64)
    pn = np.array([counts_new.get(k, 0) for k in keys], dtype=np.float64)

    pb = pb / (pb.sum() + 1e-12)
    pn = pn / (pn.sum() + 1e-12)

    return float(np.sum(pn * np.log((pn + 1e-12) / (pb + 1e-12))))

In [246]:
def entropy_of_counts(counts: dict):
    """
    Compute Shannon entropy of the category distribution.

    Interpretation
    --------------
    Higher entropy -> more diverse / spread-out categories
    Lower entropy  -> recommendations concentrated in fewer categories

    Returns
    -------
    float
        Entropy (nats).
    """
    _, p = counts_to_prob(counts)
    return float(scipy_entropy(p + 1e-12))

In [247]:
def cosine_shift(vec_before: np.ndarray, vec_after: np.ndarray):
    """
    Cosine distance between two vectors:
        1 - cos_sim(a, b)

    Interpretation
    --------------
    0.0 => identical direction
    1.0 => orthogonal
    2.0 => opposite direction (rare in practice here)

    Use-case in your story:
    - measure how much the user embedding / output vector changed after neuron intervention

    Returns
    -------
    float
        Cosine distance.
    """
    a = vec_before / (np.linalg.norm(vec_before) + 1e-12)
    b = vec_after  / (np.linalg.norm(vec_after)  + 1e-12)
    return float(1.0 - np.dot(a, b))

### Figure Functions for Visualization

In [248]:
# Gender composition bar chart (Women/Men/Neutral)
def plot_gender_composition(metrics_dict_by_condition,
                            title="Gender composition in Top-K",
                            save_path="fig_gender_composition_topk.png"):
    """
    Bar chart of women/men/neutral percentages for each condition.
    
    metrics_dict_by_condition example:
      {
        "baseline": {"women_pct":0.4,"men_pct":0.2,"neutral_pct":0.4,...},
        "amplify":  {...},
        "suppress": {...}
      }
    """

    conditions = list(metrics_dict_by_condition.keys())

    women = [metrics_dict_by_condition[c]["women_pct"] for c in conditions]
    men   = [metrics_dict_by_condition[c]["men_pct"] for c in conditions]
    neu   = [metrics_dict_by_condition[c]["neutral_pct"] for c in conditions]

    x = np.arange(len(conditions))
    width = 0.25

    plt.figure(figsize=(8,4))

    plt.bar(x - width, women, width, label="Women")
    plt.bar(x,         men,   width, label="Men")
    plt.bar(x + width, neu,   width, label="Neutral")

    plt.xticks(x, conditions)
    plt.ylim(0, 1.0)

    plt.ylabel("Fraction of Top-K items")
    plt.title(title)

    plt.grid(axis="y", alpha=0.3)
    plt.legend()

    plt.tight_layout()

    plt.savefig(save_path, dpi=250, bbox_inches="tight")
    plt.close()

    print(f"Saved: {save_path}")

In [249]:
# Women to Men Ratio
def plot_women_to_men_ratio(metrics_dict_by_condition,
                            title="Women-to-Men ratio shift",
                            save_path="fig_gender_ratio_shift.png"):
    """
    Line/bar plot of the women_to_men_ratio per condition.
    Useful because it compresses the gender story into one number.
    """

    conditions = list(metrics_dict_by_condition.keys())
    ratios = [metrics_dict_by_condition[c]["women_to_men_ratio"] for c in conditions]

    plt.figure(figsize=(7,3.5))

    plt.bar(range(len(conditions)), ratios, edgecolor="black")

    plt.xticks(range(len(conditions)), conditions)

    plt.ylabel("Women/Men ratio")
    plt.title(title)

    plt.grid(axis="y", alpha=0.3)

    plt.tight_layout()

    plt.savefig(save_path, dpi=250, bbox_inches="tight")
    plt.close()

    print(f"Saved: {save_path}")

In [250]:
# Distribution Shift Summary - KL + Entropy
def plot_distribution_shift(kl_by_condition,
                            entropy_by_condition,
                            title="Category distribution shift",
                            save_path="fig_category_distribution_shift.png"):
    """
    Two small plots: KL divergence from baseline and entropy per condition.
    
    kl_by_condition example: {"baseline":0.0,"amplify":0.12,"suppress":0.08}
    entropy_by_condition:    {"baseline":2.4,"amplify":2.1,"suppress":2.6}
    """

    conditions = list(kl_by_condition.keys())

    kl_vals = [kl_by_condition[c] for c in conditions]
    ent_vals = [entropy_by_condition[c] for c in conditions]

    plt.figure(figsize=(10,3.5))

    plt.subplot(1,2,1)

    plt.bar(range(len(conditions)), kl_vals, edgecolor="black")

    plt.xticks(range(len(conditions)), conditions)
    plt.ylabel("KL(p_new || p_base)")
    plt.title("KL from baseline")

    plt.grid(axis="y", alpha=0.3)

    plt.subplot(1,2,2)

    plt.bar(range(len(conditions)), ent_vals, edgecolor="black")

    plt.xticks(range(len(conditions)), conditions)
    plt.ylabel("Entropy")
    plt.title("Entropy of categories")

    plt.grid(axis="y", alpha=0.3)

    plt.suptitle(title)

    plt.tight_layout()

    plt.savefig(save_path, dpi=250, bbox_inches="tight")
    plt.close()

    print(f"Saved: {save_path}")

In [251]:
# How much recommendations changed: Overlap@K + Rank shift
def plot_reco_change(overlap_by_condition,
                     rankshift_by_condition,
                     title="Recommendation change vs baseline",
                     save_path="fig_recommendation_shift.png"):
    """
    Shows how much the top-K list changed under intervention.
    Overlap close to 1 => very similar
    Rank shift higher   => stronger reordering among shared items
    """

    conditions = [c for c in overlap_by_condition.keys() if c != "baseline"]

    overlap_vals = [overlap_by_condition[c] for c in conditions]
    rank_vals = [rankshift_by_condition[c] for c in conditions]

    plt.figure(figsize=(10,3.5))

    plt.subplot(1,2,1)

    plt.bar(range(len(conditions)), overlap_vals, edgecolor="black")

    plt.xticks(range(len(conditions)), conditions)
    plt.ylim(0,1.0)

    plt.ylabel("Overlap@K with baseline")
    plt.title("Set overlap")

    plt.grid(axis="y", alpha=0.3)

    plt.subplot(1,2,2)

    plt.bar(range(len(conditions)), rank_vals, edgecolor="black")

    plt.xticks(range(len(conditions)), conditions)

    plt.ylabel("Mean abs rank shift")
    plt.title("Rank shift (shared items)")

    plt.grid(axis="y", alpha=0.3)

    plt.suptitle(title)

    plt.tight_layout()

    plt.savefig(save_path, dpi=250, bbox_inches="tight")
    plt.close()

    print(f"Saved: {save_path}")

In [252]:
# Category frequency change (top category tokens)
def plot_top_category_deltas(counts_base,
                             counts_new,
                             top_n=15,
                             title="Top category token deltas",
                             save_path="fig_category_token_deltas.png"):
    """
    Plot the biggest category token increases/decreases vs baseline.

    For each token:
      delta = count_new - count_base

    Shows the strongest category moves (good for “why did gender shift?” explanation).
    """

    keys = sorted(set(counts_base) | set(counts_new))

    deltas = {k: counts_new.get(k,0) - counts_base.get(k,0) for k in keys}

    top = sorted(deltas.items(), key=lambda kv: abs(kv[1]), reverse=True)[:top_n]

    tokens = [t for t,_ in top][::-1]
    vals = [v for _,v in top][::-1]

    plt.figure(figsize=(10,5))

    plt.barh(range(len(tokens)), vals, edgecolor="black")

    plt.yticks(range(len(tokens)), tokens)

    plt.axvline(0, color="black", linewidth=1)

    plt.title(title)

    plt.xlabel("Delta count (new - baseline)")

    plt.grid(axis="x", alpha=0.3)

    plt.tight_layout()

    plt.savefig(save_path, dpi=250, bbox_inches="tight")
    plt.close()

    print(f"Saved: {save_path}")

### Run the story

#### Single User Experiment Function: Baseline vs Amplify vs Suppress

In [253]:
def run_gender_axis_experiment_for_user(
    user_idx: int,
    P: np.ndarray,
    Q: np.ndarray,
    b_u: np.ndarray,
    b_i: np.ndarray,
    mat_sae,
    dataset_items,                     # torch Tensor of item embeddings (same ordering as Q)
    women_dom_neurons: list[int],       # e.g., [3, 32, 39, 41, 45, 48]
    items_df: pd.DataFrame,
    item_gender_tag_fn,                # item_gender_tag(item_id) -> "women"/"men"/"neutral"
    recommend_topk_from_item_emb_fn,    # your recommend_topk_from_item_emb(...)
    msae_reconstruct_with_scaled_neurons_fn,  # your msae_reconstruct_with_scaled_neurons(...)
    topK: int = 50,
    amplify_scale: float = 2,
    suppress_scale: float = 0.3,
    seen: set[int] | None = None,
    categories_col: str = "categories",
):
    """
    Run Story-3 intervention for ONE user:
      - baseline: original MF item embeddings Q
      - amplify : reconstruct Q_hat with MSAE while scaling selected neurons up
      - suppress: reconstruct Q_hat with MSAE while scaling selected neurons down

    Computes:
      - gender composition metrics (women/men/neutral %, women-to-men ratio)
      - recommendation change (overlap@K, rank shift)
      - category distribution shift (KL divergence, entropy)
      - embedding-space shift (cosine distance between mean topK item vectors)

    Returns
    -------
    out : dict
        One-row metrics for this user (easy to turn into a dataframe).
    artifacts : dict
        Contains topK lists and category counts for plotting/debugging.
    """

    # -------------------------
    # 0) Baseline recommend
    # -------------------------
    base_top, _ = recommend_topk_from_item_emb_fn(P, Q, b_u, b_i, user_idx, K=topK, seen=seen)

    base_gender = gender_metrics(base_top, item_gender_tag_fn=item_gender_tag_fn)
    base_counts = category_token_counts(base_top, items_df=items_df, categories_col=categories_col)

    # -------------------------
    # 1) Amplify selected neurons
    # -------------------------
    Q_hat_amp, _ = msae_reconstruct_with_scaled_neurons_fn(
        mat_sae, dataset_items, women_dom_neurons, amplify_scale
    )
    Q_amp = Q_hat_amp.detach().cpu().numpy()

    amp_top, _ = recommend_topk_from_item_emb_fn(P, Q_amp, b_u, b_i, user_idx, K=topK, seen=seen)

    amp_gender = gender_metrics(amp_top, item_gender_tag_fn=item_gender_tag_fn)
    amp_counts = category_token_counts(amp_top, items_df=items_df, categories_col=categories_col)

    # -------------------------
    # 2) Suppress selected neurons
    # -------------------------
    Q_hat_sup, _ = msae_reconstruct_with_scaled_neurons_fn(
        mat_sae, dataset_items, women_dom_neurons, suppress_scale
    )
    Q_sup = Q_hat_sup.detach().cpu().numpy()

    sup_top, _ = recommend_topk_from_item_emb_fn(P, Q_sup, b_u, b_i, user_idx, K=topK, seen=seen)

    sup_gender = gender_metrics(sup_top, item_gender_tag_fn=item_gender_tag_fn)
    sup_counts = category_token_counts(sup_top, items_df=items_df, categories_col=categories_col)

    # -------------------------
    # 3) Extra metrics (set/rank/category/embedding shifts)
    # -------------------------
    out = {
        "user": int(user_idx),
        "topK": int(topK),

        # --- gender composition
        "base_women_pct": base_gender["women_pct"],
        "amp_women_pct":  amp_gender["women_pct"],
        "sup_women_pct":  sup_gender["women_pct"],

        "base_men_pct": base_gender["men_pct"],
        "amp_men_pct":  amp_gender["men_pct"],
        "sup_men_pct":  sup_gender["men_pct"],

        "base_neutral_pct": base_gender["neutral_pct"],
        "amp_neutral_pct":  amp_gender["neutral_pct"],
        "sup_neutral_pct":  sup_gender["neutral_pct"],

        "base_ratio": base_gender["women_to_men_ratio"],
        "amp_ratio":  amp_gender["women_to_men_ratio"],
        "sup_ratio":  sup_gender["women_to_men_ratio"],

        # --- recommendation change vs baseline
        "overlap_base_amp": overlap_at_k(base_top, amp_top),
        "overlap_base_sup": overlap_at_k(base_top, sup_top),

        "rank_shift_base_amp": rank_shift(base_top, amp_top),
        "rank_shift_base_sup": rank_shift(base_top, sup_top),

        # --- category distribution shift vs baseline
        "kl_cat_base_amp": kl_divergence_from_baseline(base_counts, amp_counts),
        "kl_cat_base_sup": kl_divergence_from_baseline(base_counts, sup_counts),

        "entropy_base": entropy_of_counts(base_counts),
        "entropy_amp":  entropy_of_counts(amp_counts),
        "entropy_sup":  entropy_of_counts(sup_counts),

        # --- embedding-space shift (mean topK item vectors)
        "cos_shift_topk_mean_base_amp": cosine_shift(Q[base_top].mean(axis=0), Q_amp[amp_top].mean(axis=0)),
        "cos_shift_topk_mean_base_sup": cosine_shift(Q[base_top].mean(axis=0), Q_sup[sup_top].mean(axis=0)),
    }

    artifacts = {
        "base_top": base_top,
        "amp_top": amp_top,
        "sup_top": sup_top,
        "base_counts": base_counts,
        "amp_counts": amp_counts,
        "sup_counts": sup_counts,
    }

    return out, artifacts

#### Run on Many Users

In [254]:
def run_gender_axis_experiment_many_users(
    users: np.ndarray,
    P: np.ndarray,
    Q: np.ndarray,
    b_u: np.ndarray,
    b_i: np.ndarray,
    mat_sae,
    dataset_items,
    women_dom_neurons: list[int],
    items_df: pd.DataFrame,
    item_gender_tag_fn,
    recommend_topk_from_item_emb_fn,
    msae_reconstruct_with_scaled_neurons_fn,
    topK: int = 50,
    amplify_scale: float = 2,
    suppress_scale: float = 0.3,
    categories_col: str = "categories",
    seen_from_R_train=None,  # optional: function or matrix to get seen items per user
):
    """
    Run Story-3 for MANY users and aggregate results.

    Returns
    -------
    df : pd.DataFrame
        One row per user with all scalar metrics.
    agg : dict
        Aggregated metrics and category counts for plotting:
        - gender_metrics_by_condition
        - kl_by_condition
        - entropy_by_condition
        - overlap_by_condition
        - rankshift_by_condition
        - category_counts_by_condition
    """

    rows = []
    # Aggregate category counts across users (sum counts)
    base_counts_total = Counter()
    amp_counts_total  = Counter()
    sup_counts_total  = Counter()

    for u in users:
        u = int(u)

        # optional: define seen items per user (so we don't recommend already-seen)
        seen = None
        if seen_from_R_train is not None:
            # either a CSR matrix R_train or a callable
            if callable(seen_from_R_train):
                seen = set(map(int, seen_from_R_train(u)))
            else:
                seen = set(map(int, seen_from_R_train[u].indices))

        out, art = run_gender_axis_experiment_for_user(
            user_idx=u,
            P=P, Q=Q, b_u=b_u, b_i=b_i,
            mat_sae=mat_sae,
            dataset_items=dataset_items,
            women_dom_neurons=women_dom_neurons,
            items_df=items_df,
            item_gender_tag_fn=item_gender_tag_fn,
            recommend_topk_from_item_emb_fn=recommend_topk_from_item_emb_fn,
            msae_reconstruct_with_scaled_neurons_fn=msae_reconstruct_with_scaled_neurons_fn,
            topK=topK,
            amplify_scale=amplify_scale,
            suppress_scale=suppress_scale,
            seen=seen,
            categories_col=categories_col,
        )

        rows.append(out)
        base_counts_total.update(art["base_counts"])
        amp_counts_total.update(art["amp_counts"])
        sup_counts_total.update(art["sup_counts"])

    df = pd.DataFrame(rows)

    # ---- Aggregate scalar metrics (mean over users) ----
    gender_metrics_by_condition = {
        "baseline": {
            "women_pct": float(df["base_women_pct"].mean()),
            "men_pct": float(df["base_men_pct"].mean()),
            "neutral_pct": float(df["base_neutral_pct"].mean()),
            "women_to_men_ratio": float(df["base_ratio"].mean()),
        },
        "amplify": {
            "women_pct": float(df["amp_women_pct"].mean()),
            "men_pct": float(df["amp_men_pct"].mean()),
            "neutral_pct": float(df["amp_neutral_pct"].mean()),
            "women_to_men_ratio": float(df["amp_ratio"].mean()),
        },
        "suppress": {
            "women_pct": float(df["sup_women_pct"].mean()),
            "men_pct": float(df["sup_men_pct"].mean()),
            "neutral_pct": float(df["sup_neutral_pct"].mean()),
            "women_to_men_ratio": float(df["sup_ratio"].mean()),
        },
    }

    kl_by_condition = {
        "baseline": 0.0,
        "amplify": float(df["kl_cat_base_amp"].mean()),
        "suppress": float(df["kl_cat_base_sup"].mean()),
    }

    entropy_by_condition = {
        "baseline": float(df["entropy_base"].mean()),
        "amplify": float(df["entropy_amp"].mean()),
        "suppress": float(df["entropy_sup"].mean()),
    }

    overlap_by_condition = {
        "baseline": 1.0,
        "amplify": float(df["overlap_base_amp"].mean()),
        "suppress": float(df["overlap_base_sup"].mean()),
    }

    rankshift_by_condition = {
        "baseline": 0.0,
        "amplify": float(df["rank_shift_base_amp"].mean()),
        "suppress": float(df["rank_shift_base_sup"].mean()),
    }

    agg = {
        "gender_metrics_by_condition": gender_metrics_by_condition,
        "kl_by_condition": kl_by_condition,
        "entropy_by_condition": entropy_by_condition,
        "overlap_by_condition": overlap_by_condition,
        "rankshift_by_condition": rankshift_by_condition,
        "category_counts_by_condition": {
            "baseline": dict(base_counts_total),
            "amplify": dict(amp_counts_total),
            "suppress": dict(sup_counts_total),
        }
    }

    return df, agg

Plotting the 5 figures using the aggregated output

In [255]:
def plot_story3_all_figures(agg: dict, top_n_deltas: int = 15):
    """
    Produce the 5 figures for Story-3 from aggregated results.

    Figures:
      1) gender composition (women/men/neutral)
      2) women-to-men ratio
      3) KL + entropy
      4) overlap + rank shift
      5) category token deltas (baseline vs amplify and baseline vs suppress)
    """
    gm = agg["gender_metrics_by_condition"]
    kl = agg["kl_by_condition"]
    ent = agg["entropy_by_condition"]
    ov = agg["overlap_by_condition"]
    rs = agg["rankshift_by_condition"]
    counts = agg["category_counts_by_condition"]

    # 1) Gender composition
    plot_gender_composition(gm, title="Story-3: Gender composition in Top-K (aggregated)")

    # 2) Ratio
    plot_women_to_men_ratio(gm, title="Story-3: Women-to-Men ratio shift (aggregated)")

    # 3) KL + Entropy
    plot_distribution_shift(kl, ent, title="Story-3: Category distribution shift (aggregated)")

    # 4) Overlap + Rank shift
    plot_reco_change(ov, rs, title="Story-3: Recommendation change vs baseline (aggregated)")

    # 5) Category deltas (two plots)
    plot_top_category_deltas(
        counts_base=counts["baseline"],
        counts_new=counts["amplify"],
        top_n=top_n_deltas,
        title="Story-3: Top category token deltas (Amplify vs Baseline)"
    )
    plot_top_category_deltas(
        counts_base=counts["baseline"],
        counts_new=counts["suppress"],
        top_n=top_n_deltas,
        title="Story-3: Top category token deltas (Suppress vs Baseline)"
    )

#### Usage

One User

In [256]:
u = 0
seen = set(R_train[u].indices)  # if you have R_train

row_u, artifacts_u = run_gender_axis_experiment_for_user(
    user_idx=u,
    P=P, Q=Q, b_u=b_u, b_i=b_i,
    mat_sae=mat_sae,
    dataset_items=dataset_items,
    women_dom_neurons=WOMEN_DOM_NEURONS,
    items_df=items_df,
    item_gender_tag_fn=item_gender_tag,
    recommend_topk_from_item_emb_fn=recommend_topk_from_item_emb,
    msae_reconstruct_with_scaled_neurons_fn=msae_reconstruct_with_scaled_neurons,
    topK=50,
    amplify_scale=1.5,
    suppress_scale=0.5,
    seen=seen,
)

row_u

{'user': 0,
 'topK': 50,
 'base_women_pct': 0.56,
 'amp_women_pct': 0.56,
 'sup_women_pct': 0.56,
 'base_men_pct': 0.28,
 'amp_men_pct': 0.28,
 'sup_men_pct': 0.28,
 'base_neutral_pct': 0.16,
 'amp_neutral_pct': 0.16,
 'sup_neutral_pct': 0.16,
 'base_ratio': 1.999999996428571,
 'amp_ratio': 1.999999996428571,
 'sup_ratio': 1.999999996428571,
 'overlap_base_amp': 0.98,
 'overlap_base_sup': 0.98,
 'rank_shift_base_amp': 0.8775510204081632,
 'rank_shift_base_sup': 1.0204081632653061,
 'kl_cat_base_amp': 0.005641413841633532,
 'kl_cat_base_sup': 0.005641413841633532,
 'entropy_base': 3.3082056718088997,
 'entropy_amp': 3.2933715836979895,
 'entropy_sup': 3.2933715836979895,
 'cos_shift_topk_mean_base_amp': 0.12016475200653076,
 'cos_shift_topk_mean_base_sup': 0.15490758419036865}

Many Users

In [257]:
np.random.seed(42)
sample_users = np.random.choice(np.arange(P.shape[0]), size=50, replace=False)

df_story3, agg_story3 = run_gender_axis_experiment_many_users(
    users=sample_users,
    P=P, Q=Q, b_u=b_u, b_i=b_i,
    mat_sae=mat_sae,
    dataset_items=dataset_items,
    women_dom_neurons=WOMEN_DOM_NEURONS,
    items_df=items_df,
    item_gender_tag_fn=item_gender_tag,
    recommend_topk_from_item_emb_fn=recommend_topk_from_item_emb,
    msae_reconstruct_with_scaled_neurons_fn=msae_reconstruct_with_scaled_neurons,
    topK=50,
    amplify_scale=1.5,
    suppress_scale=0.5,
    seen_from_R_train=R_train,  # pass R_train to exclude seen items
)

df_story3.describe()
plot_story3_all_figures(agg_story3, top_n_deltas=15)

Saved: fig_gender_composition_topk.png
Saved: fig_gender_ratio_shift.png
Saved: fig_category_distribution_shift.png
Saved: fig_recommendation_shift.png
Saved: fig_category_token_deltas.png
Saved: fig_category_token_deltas.png


In [258]:
df_story3

,user,topK,base_women_pct,amp_women_pct,sup_women_pct,base_men_pct,amp_men_pct,sup_men_pct,base_neutral_pct,amp_neutral_pct,...,overlap_base_sup,rank_shift_base_amp,rank_shift_base_sup,kl_cat_base_amp,kl_cat_base_sup,entropy_base,entropy_amp,entropy_sup,cos_shift_topk_mean_base_amp,cos_shift_topk_mean_base_sup
0,23218,50,0.56,0.54,0.54,0.30,0.30,0.30,0.14,0.16,...,0.96,0.829787,1.020833,0.100059,0.093041,3.298490,3.308923,3.307272,0.127762,0.051984
1,20731,50,0.52,0.52,0.54,0.30,0.30,0.30,0.18,0.18,...,0.96,1.020408,0.916667,0.100531,0.113699,3.338056,3.320272,3.274507,0.099717,0.133263
2,39555,50,0.52,0.52,0.52,0.32,0.32,0.32,0.16,0.16,...,1.00,1.040000,1.040000,0.000000,0.000000,3.347388,3.347388,3.347388,0.081750,0.073621
3,147506,50,0.52,0.50,0.50,0.32,0.34,0.34,0.16,0.16,...,0.98,0.877551,0.979592,0.096460,0.096460,3.273677,3.272672,3.272672,0.123261,0.138168
4,314215,50,0.54,0.52,0.52,0.30,0.32,0.32,0.16,0.16,...,0.98,1.489796,1.469388,0.005593,0.008531,3.279107,3.283597,3.277222,0.078289,0.042148
5,190913,50,0.52,0.52,0.52,0.32,0.32,0.32,0.16,0.16,...,0.98,1.040000,1.000000,0.000000,0.006368,3.290188,3.290188,3.270746,0.100250,0.124197
6,296715,50,0.52,0.48,0.50,0.30,0.34,0.34,0.18,0.18,...,0.94,1.125000,1.212766,0.182738,0.276602,3.303912,3.342448,3.335762,0.144442,0.128076
7,141482,50,0.54,0.54,0.54,0.32,0.32,0.32,0.14,0.14,...,0.98,0.816327,0.938776,0.015722,0.015722,3.324697,3.286320,3.286320,0.117129,0.100384
8,49119,50,0.48,0.50,0.50,0.34,0.34,0.34,0.18,0.16,...,0.98,1.142857,1.204082,0.096371,0.096371,3.341384,3.331951,3.331951,0.123228,0.090857
9,208005,50,0.54,0.52,0.52,0.30,0.32,0.32,0.16,0.16,...,0.98,1.285714,1.244898,0.005844,0.005844,3.280347,3.263495,3.263495,0.063504,0.075866


Intervention Strength vs Recommendation Outcome

In [259]:
scales = [2.0, 4.0, 8.0, 16.0]

results = []

for s in scales:

    df_tmp, agg_tmp = run_gender_axis_experiment_many_users(
        users=sample_users,
        P=P, Q=Q, b_u=b_u, b_i=b_i,
        mat_sae=mat_sae,
        dataset_items=dataset_items,
        women_dom_neurons=WOMEN_DOM_NEURONS,
        items_df=items_df,
        item_gender_tag_fn=item_gender_tag,
        recommend_topk_from_item_emb_fn=recommend_topk_from_item_emb,
        msae_reconstruct_with_scaled_neurons_fn=msae_reconstruct_with_scaled_neurons,
        topK=50,
        amplify_scale=s,
        suppress_scale=0.1,
        seen_from_R_train=R_train
    )

    results.append({
        "scale": s,
        "women_pct": df_tmp["amp_women_pct"].mean(),
        "men_pct": df_tmp["amp_men_pct"].mean()
    })

# Plot 
df_curve = pd.DataFrame(results)

plt.figure(figsize=(6,5))

plt.plot(df_curve["scale"], df_curve["women_pct"], marker="o", label="Women")
plt.plot(df_curve["scale"], df_curve["men_pct"], marker="o", label="Men")

plt.axvline(1.0, linestyle="--", color="gray", label="Baseline")

plt.xlabel("Neuron Scaling Factor")
plt.ylabel("Average Recommendation Share")

plt.title("Effect of Neuron Scaling on Recommendation Gender Distribution")

plt.legend()

plt.grid(alpha=0.3)

plt.savefig("intervention_response_curve.png", dpi=250, bbox_inches="tight")

plt.close()